In [ ]:
# CELL 0: Mount Drive and configure E08 SAM2 training workspace
from google.colab import drive
from IPython.display import Image, display
from pathlib import Path
import os, re, shutil, subprocess, sys, json, glob, time, hashlib

drive.mount('/content/drive')

# ========================== EDIT THIS LINE ==========================
# This label is a Drive folder name and must also be used by inference.
MICROSCOPE_LABEL = "E08"
# ======================== END EDIT LINE ========================

RELEASE_NOTEBOOK_VERSION = "syniscopy-sam2-public-release-2026-05-15"

def sanitize_label(label: str) -> str:
    clean = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(label).strip())
    return clean.strip("._-") or "default_configuration"

MICROSCOPE_LABEL = sanitize_label(MICROSCOPE_LABEL)
SOURCE_ZIP_NAME = 'syniscopy_source.zip'
DRIVE_MYDRIVE = Path('/content/drive/MyDrive')


def find_supplemental_root() -> Path:
    explicit = globals().get('SYNISCOPY_SUPPLEMENTAL_ROOT', None)
    candidates = []
    if explicit:
        candidates.append(Path(explicit).expanduser())
    if DRIVE_MYDRIVE.exists():
        candidates.extend([
            DRIVE_MYDRIVE / 'supplemental',
            DRIVE_MYDRIVE / 'SyniscopySupplemental',
        ])
    candidates.extend([Path('/content/supplemental'), Path('/content/SyniscopySupplemental')])
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if (resolved / 'supplemental' / 'E01.ipynb').exists():
            resolved = resolved / 'supplemental'
        if (resolved / 'E01.ipynb').exists():
            return resolved
    raise RuntimeError('Upload the supplemental folder to Drive as MyDrive/supplemental, including syniscopy_source.zip.')


SUPPLEMENTAL_ROOT = find_supplemental_root()
DRIVE_ROOT = SUPPLEMENTAL_ROOT
OUTPUT_ROOT = SUPPLEMENTAL_ROOT / 'outputs'
CONFIG_DIR = OUTPUT_ROOT / MICROSCOPE_LABEL
RAW_DATASET_ROOT = OUTPUT_ROOT / 'E07' / 'dataset' / 'raw_syniscopy'
SAM2_DATASET_ROOT = OUTPUT_ROOT / 'E07' / 'dataset' / 'sam2_vos'
WEIGHTS_DIR = CONFIG_DIR / 'weights'
TRAINING_LOG_DIR = CONFIG_DIR / 'training_logs'
TRAINING_CHECKPOINT_DIR = TRAINING_LOG_DIR / 'checkpoints'
FINAL_CHECKPOINT_PATH = WEIGHTS_DIR / 'final_checkpoint.pt'
CHECKPOINT_MANIFEST_PATH = WEIGHTS_DIR / 'checkpoint_manifest.json'

SOURCE_ZIP_CANDIDATES = [SUPPLEMENTAL_ROOT / SOURCE_ZIP_NAME]
SOURCE_ZIP_PATH = next((p for p in SOURCE_ZIP_CANDIDATES if p.exists()), SOURCE_ZIP_CANDIDATES[0])
SYNISCOPY_EXTRACT_DIR = Path('/content/syniscopy_source')

for d in [DRIVE_ROOT, CONFIG_DIR, WEIGHTS_DIR, TRAINING_LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Notebook version:', RELEASE_NOTEBOOK_VERSION)
print('Supplemental root:', SUPPLEMENTAL_ROOT)
print('Microscope/config label:', MICROSCOPE_LABEL)
print('Raw dataset:', RAW_DATASET_ROOT)
print('SAM2 dataset:', SAM2_DATASET_ROOT)
print('Final checkpoint:', FINAL_CHECKPOINT_PATH)
print('Expected source zip:', SOURCE_ZIP_PATH)

print('E08 trains from the E07-owned SAM2 VOS cache; weights/logs are under the E08 label.')


In [ ]:
# CELL 0.5: Put Syniscopy source on Python path
import zipfile

MOUNTED_REPO_CANDIDATES = [
    SUPPLEMENTAL_ROOT.parent,
    Path('/content/SYNISCOPY'),
    Path('/content/syniscopy'),
]
SYNISCOPY_REPO_DIR = next((p for p in MOUNTED_REPO_CANDIDATES if (p / 'codebase').is_dir()), None)

if SYNISCOPY_REPO_DIR is None:
    if not SOURCE_ZIP_PATH.exists():
        raise FileNotFoundError(
            f'Could not find a mounted Syniscopy repo and no source zip exists at {SOURCE_ZIP_PATH}. '
            'Upload supplemental/ with syniscopy_source.zip, or run from a repo containing codebase/.'
        )
    if SYNISCOPY_EXTRACT_DIR.exists():
        shutil.rmtree(SYNISCOPY_EXTRACT_DIR)
    SYNISCOPY_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(SOURCE_ZIP_PATH) as zf:
        zf.extractall(SYNISCOPY_EXTRACT_DIR)
    if (SYNISCOPY_EXTRACT_DIR / 'codebase').is_dir():
        SYNISCOPY_REPO_DIR = SYNISCOPY_EXTRACT_DIR
    else:
        candidates = [p for p in SYNISCOPY_EXTRACT_DIR.iterdir() if (p / 'codebase').is_dir()]
        if len(candidates) != 1:
            raise RuntimeError('Could not find codebase/ inside extracted Syniscopy source zip.')
        SYNISCOPY_REPO_DIR = candidates[0]
    print('Syniscopy source zip:', SOURCE_ZIP_PATH)
else:
    print('Using mounted Syniscopy repo:', SYNISCOPY_REPO_DIR)

CODEBASE_DIR = SYNISCOPY_REPO_DIR / 'codebase'
SAM2_STARTER_DIR = SYNISCOPY_REPO_DIR / 'sam2_starter'
for required in ['config.py', 'dataset_generator.py', 'dataset_schema.py', 'main.py', 'postprocessing.py']:
    if not (CODEBASE_DIR / required).exists():
        raise FileNotFoundError(f'Missing required Syniscopy codebase file: {CODEBASE_DIR / required}')

for path in [SYNISCOPY_REPO_DIR, CODEBASE_DIR, SAM2_STARTER_DIR]:
    if path.exists():
        path_str = str(path)
        if path_str in sys.path:
            sys.path.remove(path_str)
        sys.path.insert(0, path_str)

# Runtime dependencies for Syniscopy generation/conversion. Torch/SAM2 setup happens below.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless>=4.8,<4.12', 'tqdm', 'numpy<2', 'scipy'], check=False)

# Force this runtime to import modules from the mounted/extracted source.
import importlib
SYNISCOPY_MODULES = [
    'camera_noise', 'config', 'counterfactual_packets', 'dataset_generator',
    'dataset_schema', 'imaging_base', 'imaging_model', 'kohler_imaging',
    'main', 'mask_generation', 'materials', 'metadata', 'optics',
    'param_schema', 'param_utils', 'particle_model', 'particle_specs',
    'postprocessing', 'presets', 'pupil_sampling', 'rendering',
    'single_frame_viewer', 'substrate', 'substrate_fitting', 'substrate_pattern',
    'supervision_policy', 'trackability', 'trajectory'
]
for name in SYNISCOPY_MODULES:
    if name in sys.modules:
        del sys.modules[name]
print('Syniscopy codebase:', CODEBASE_DIR)


## Optional single-frame preview

Use this section to tune what the generated microscopy data should look like before spending time on a full dataset. The common controls are exposed as Colab form fields. For any advanced simulator key, add it to `CUSTOM_PARAM_OVERRIDES` in the preview cell; the full resolved parameter JSON is saved with every preview.

You can skip this section if you already know the generation parameters you want to train on.


In [ ]:
# CELL 0.6: Optional single-frame preview / tuning cell
# @title Single-frame preview controls
import json
import numpy as np
from IPython.display import Image, display
from dataset_generator import build_dataset_video_params, get_default_dataset_params
from single_frame_viewer import save_single_frame_preview

RUN_SINGLE_FRAME_PREVIEW = False  # @param {type:"boolean"}

# The preview uses the public default recipe. Edit concrete microscope/sample
# parameters below for one-frame inspection only.
PREVIEW_DATASET_PRESET_NAME = "default"
PREVIEW_DATASET_INSTRUMENT_PRESET = ""  # @param {type:"string"}
PREVIEW_RANDOM_SEED = 12345  # @param {type:"integer"}

# @markdown **Image and particle**
IMAGE_SIZE_PIXELS = 512  # @param {type:"integer"}
PIXEL_SIZE_NM = 100.0  # @param {type:"number"}
NUM_PARTICLES = 1  # @param {type:"integer"}
PARTICLE_DIAMETER_NM = 100.0  # @param {type:"number"}
PARTICLE_HYDRODYNAMIC_DIAMETER_NM = 100.0  # @param {type:"number"}
PARTICLE_MATERIAL = "polystyrene"  # @param ["polystyrene", "gold", "silver", "silica", "protein"]
PARTICLE_SIGNAL_MULTIPLIER = 1.0  # @param {type:"number"}

# @markdown **Optics and modality**
IMAGING_MODEL = "bright_field"  # @param ["bright_field", "fluorescence_widefield", "tirf_fluorescence", "dark_field", "zernike_phase_contrast", "differential_phase_contrast", "quantitative_phase", "off_axis_holography", "ricm", "interferometric", "partially_coherent_bright_field", "coherent_bright_field", "coherent_dark_field"]
WAVELENGTH_NM = 550.0  # @param {type:"number"}
NUMERICAL_APERTURE = 0.8  # @param {type:"number"}
REFRACTIVE_INDEX_MEDIUM = 1.33  # @param {type:"number"}
REFRACTIVE_INDEX_IMMERSION = 1.33  # @param {type:"number"}

# @markdown **Fluorescence material properties**
PARTICLE_FLUOROPHORE_DENSITY = 1.0  # @param {type:"number"}
FLUORESCENCE_EXCITATION_WAVELENGTH_NM = 488.0  # @param {type:"number"}
FLUORESCENCE_EMISSION_WAVELENGTH_NM = 520.0  # @param {type:"number"}

# @markdown **Background and noise**
BACKGROUND_INTENSITY = 100.0  # @param {type:"number"}
CAMERA_GAIN_E_PER_COUNT = 1.0  # @param {type:"number"}
READ_NOISE_COUNTS = 1.0  # @param {type:"number"}
SHOT_NOISE_ENABLED = True  # @param {type:"boolean"}
GAUSSIAN_NOISE_ENABLED = True  # @param {type:"boolean"}
EMPIRICAL_BACKGROUND_ENABLED = False  # @param {type:"boolean"}
SAMPLE_ENVIRONMENT_PATTERN_ENABLED = False  # @param {type:"boolean"}

# @markdown **Masks**
MASK_OUTER_RING_COUNT = 0  # @param {type:"integer"}
SUPERVISION_TARGET_FOR_TRAINING = "mask_supported"  # @param ["mask_supported", "mask_geometry"]

# Advanced: add any full config.py PARAMS key here. These values override the
# form fields above. Use canonical particle objects under "particles".
CUSTOM_PARAM_OVERRIDES = {}


def _sphere_particle(index, material, diameter_nm, hydrodynamic_diameter_nm, signal_multiplier, material_properties=None):
    return {
        'name': f'particle_{index}',
        'motion': {
            'hydrodynamic_diameter_nm': float(hydrodynamic_diameter_nm),
            'initial_position_nm': None,
        },
        'signal_multiplier': float(signal_multiplier),
        'components': [
            {
                'shape': 'sphere',
                'offset_nm': [0.0, 0.0, 0.0],
                'diameter_nm': float(diameter_nm),
                'material': str(material),
                'refractive_index': None,
                'signal_multiplier': 1.0,
                'material_properties': material_properties,
            }
        ],
    }


def _preview_particles():
    material_properties = None
    if IMAGING_MODEL in {'fluorescence_widefield', 'tirf_fluorescence'}:
        material_properties = {
            'fluorophore_density': float(PARTICLE_FLUOROPHORE_DENSITY),
            'excitation_peak_nm': float(FLUORESCENCE_EXCITATION_WAVELENGTH_NM),
            'emission_peak_nm': float(FLUORESCENCE_EMISSION_WAVELENGTH_NM),
        }
    return [
        _sphere_particle(
            index=i,
            material=PARTICLE_MATERIAL,
            diameter_nm=PARTICLE_DIAMETER_NM,
            hydrodynamic_diameter_nm=PARTICLE_HYDRODYNAMIC_DIAMETER_NM,
            signal_multiplier=PARTICLE_SIGNAL_MULTIPLIER,
            material_properties=material_properties,
        )
        for i in range(int(NUM_PARTICLES))
    ]

if RUN_SINGLE_FRAME_PREVIEW:
    preview_dir = CONFIG_DIR / 'preview'
    preview_dir.mkdir(parents=True, exist_ok=True)

    common_overrides = {
        'image_size_pixels': int(IMAGE_SIZE_PIXELS),
        'pixel_size_nm': float(PIXEL_SIZE_NM),
        'particles': _preview_particles(),
        'imaging_model': IMAGING_MODEL,
        'wavelength_nm': float(WAVELENGTH_NM),
        'numerical_aperture': float(NUMERICAL_APERTURE),
        'refractive_index_medium': float(REFRACTIVE_INDEX_MEDIUM),
        'refractive_index_immersion': float(REFRACTIVE_INDEX_IMMERSION),
        'background_intensity': float(BACKGROUND_INTENSITY),
        'camera_gain_e_per_count': float(CAMERA_GAIN_E_PER_COUNT),
        'read_noise_counts': float(READ_NOISE_COUNTS),
        'shot_noise_enabled': bool(SHOT_NOISE_ENABLED),
        'gaussian_noise_enabled': bool(GAUSSIAN_NOISE_ENABLED),
        'empirical_background_enabled': bool(EMPIRICAL_BACKGROUND_ENABLED),
        'sample_environment_pattern_enabled': bool(SAMPLE_ENVIRONMENT_PATTERN_ENABLED),
        'sample_environment_pattern': 'gold_holes' if SAMPLE_ENVIRONMENT_PATTERN_ENABLED else 'none',
        'mask_outer_ring_count': int(MASK_OUTER_RING_COUNT),
        'supervision_target': SUPERVISION_TARGET_FOR_TRAINING,
    }

    if IMAGING_MODEL in {'fluorescence_widefield', 'tirf_fluorescence'}:
        common_overrides.update({
            'fluorescence_excitation_wavelength_nm': float(FLUORESCENCE_EXCITATION_WAVELENGTH_NM),
            'fluorescence_emission_wavelength_nm': float(FLUORESCENCE_EMISSION_WAVELENGTH_NM),
        })

    default_surface = get_default_dataset_params()
    unknown_custom_keys = sorted(set(CUSTOM_PARAM_OVERRIDES) - set(default_surface))
    if unknown_custom_keys:
        raise ValueError(f'CUSTOM_PARAM_OVERRIDES contains unsupported PARAMS key(s): {unknown_custom_keys}')

    PREVIEW_PARAM_OVERRIDES = {**common_overrides, **CUSTOM_PARAM_OVERRIDES}

    template_path = preview_dir / 'complete_default_parameter_surface.json'
    with open(template_path, 'w', encoding='utf-8') as fh:
        json.dump(default_surface, fh, indent=2, sort_keys=True, default=str)

    master_rng = np.random.default_rng(PREVIEW_RANDOM_SEED)
    preview_seed = int(master_rng.integers(0, 2 ** 31))
    preview_rng = np.random.default_rng(preview_seed)
    instrument_preset = PREVIEW_DATASET_INSTRUMENT_PRESET or None
    preview_params = build_dataset_video_params(
        video_index=0,
        rng=preview_rng,
        preset_name=PREVIEW_DATASET_PRESET_NAME,
        instrument_preset=instrument_preset,
        param_overrides=PREVIEW_PARAM_OVERRIDES,
    )
    preview_outputs = save_single_frame_preview(preview_params, str(preview_dir), seed=preview_seed)
    print('Preview outputs:')
    for k, v in preview_outputs.items():
        print(f'  {k}: {v}')
    print('Complete default parameter surface:', template_path)

    for view_key in ['raw_signal_frame', 'raw_reference_frame', 'contrast_frame', 'final_frame_8bit']:
        path = preview_outputs.get(view_key)
        if path:
            print(f'Preview view: {view_key}')
            display(Image(filename=path))

    print('\nPreview only. E07 controls the full raw dataset used by this supplemental training copy.')
    print('For E08, edit SUPERVISION_TARGET or SAM2 training controls in the Final training condition cell.')
else:
    print('Preview skipped. Set RUN_SINGLE_FRAME_PREVIEW = True to render one frame.')


## Final training condition

E07 owns the raw Syniscopy dataset and the derived SAM2 VOS cache. E08 consumes that cache and trains the E08 checkpoint.

Edit the SAM2 training controls here, or keep the defaults and run the notebook top to bottom. The E07 raw dataset is read from `MyDrive/supplemental/outputs/E07/dataset/raw_syniscopy/`, the E07-owned SAM2 cache is read from `MyDrive/supplemental/outputs/E07/dataset/sam2_vos/`, and E08 weights/logs are stored under `MyDrive/supplemental/outputs/E08/`.


In [ ]:
import json
from dataset_schema import validate_supervision_target
# CELL 0.7: Final training condition for E07-owned cache consumption and E08 training.

MICROSCOPE_LABEL = "E08"
SUPERVISION_TARGET = "mask_supported"
RESET_SAM2_DATASET = False
RESET_TRAINING_CHECKPOINTS = False
CHECKPOINT_RECOVERY_ONLY = False
DATASET_ONLY = False
EMPTY_GOOGLE_DRIVE_TRASH_ON_CLEANUP = True

# SAM2 cache contract and video-training parameters. These values must match
# the E07-owned cache; E08 validates the cache instead of rebuilding it.
SAM2_TRAIN_NUM_FRAMES = 3
SAM2_MAX_NUM_OBJECTS = 3
SAM2_REVERSE_TIME_PROB = 0.5
SAM2_VAL_FRACTION = 0.15
SAM2_VAL_RANDOM_SEED = 42
SAM2_LOSS_SEMANTICS_VERSION = 'loss_weight_positive_target_v2'
SAM2_MIN_VALID_FRAMES = SAM2_TRAIN_NUM_FRAMES

_external_condition_json = globals().get("SYNISCOPY_TRAINING_CONDITION_JSON")
if _external_condition_json:
    with open(_external_condition_json, "r", encoding="utf-8") as _fh:
        _external_condition = json.load(_fh)
    for _key, _value in _external_condition.items():
        if _key in {
            "MICROSCOPE_LABEL",
            "SUPERVISION_TARGET",
            "RESET_SAM2_DATASET",
            "RESET_TRAINING_CHECKPOINTS",
            "CHECKPOINT_RECOVERY_ONLY",
            "DATASET_ONLY",
            "EMPTY_GOOGLE_DRIVE_TRASH_ON_CLEANUP",
            "SAM2_TRAIN_NUM_FRAMES",
            "SAM2_MAX_NUM_OBJECTS",
            "SAM2_REVERSE_TIME_PROB",
            "SAM2_VAL_FRACTION",
            "SAM2_VAL_RANDOM_SEED",
            "SAM2_LOSS_SEMANTICS_VERSION",
            "SAM2_MIN_VALID_FRAMES",
        }:
            globals()[_key] = _value
    print("Loaded external training condition:", _external_condition_json)

MICROSCOPE_LABEL = sanitize_label(MICROSCOPE_LABEL)
SUPERVISION_TARGET = validate_supervision_target(SUPERVISION_TARGET)
SAM2_TRAIN_NUM_FRAMES = int(SAM2_TRAIN_NUM_FRAMES)
SAM2_MAX_NUM_OBJECTS = int(SAM2_MAX_NUM_OBJECTS)
SAM2_REVERSE_TIME_PROB = float(SAM2_REVERSE_TIME_PROB)
SAM2_VAL_FRACTION = float(SAM2_VAL_FRACTION)
SAM2_VAL_RANDOM_SEED = int(SAM2_VAL_RANDOM_SEED)
SAM2_MIN_VALID_FRAMES = int(SAM2_MIN_VALID_FRAMES)
if SAM2_TRAIN_NUM_FRAMES < 1:
    raise ValueError('SAM2_TRAIN_NUM_FRAMES must be >= 1')
if SAM2_MAX_NUM_OBJECTS < 1:
    raise ValueError('SAM2_MAX_NUM_OBJECTS must be >= 1')
if not 0.0 <= SAM2_REVERSE_TIME_PROB <= 1.0:
    raise ValueError('SAM2_REVERSE_TIME_PROB must be between 0 and 1')
if not 0.0 < SAM2_VAL_FRACTION < 0.5:
    raise ValueError('SAM2_VAL_FRACTION must be > 0 and < 0.5')
if SAM2_MIN_VALID_FRAMES < SAM2_TRAIN_NUM_FRAMES:
    raise ValueError('SAM2_MIN_VALID_FRAMES must be >= SAM2_TRAIN_NUM_FRAMES')
CHECKPOINT_RECOVERY_ONLY = bool(CHECKPOINT_RECOVERY_ONLY)
if CHECKPOINT_RECOVERY_ONLY and RESET_TRAINING_CHECKPOINTS:
    raise ValueError('CHECKPOINT_RECOVERY_ONLY cannot be combined with RESET_TRAINING_CHECKPOINTS=True')

OUTPUT_ROOT = SUPPLEMENTAL_ROOT / 'outputs'
CONFIG_DIR = OUTPUT_ROOT / MICROSCOPE_LABEL
RAW_DATASET_ROOT = OUTPUT_ROOT / 'E07' / 'dataset' / 'raw_syniscopy'
SAM2_DATASET_ROOT = OUTPUT_ROOT / 'E07' / 'dataset' / 'sam2_vos'
WEIGHTS_DIR = CONFIG_DIR / 'weights'
TRAINING_LOG_DIR = CONFIG_DIR / 'training_logs'
TRAINING_CHECKPOINT_DIR = TRAINING_LOG_DIR / 'checkpoints'
FINAL_CHECKPOINT_PATH = WEIGHTS_DIR / 'final_checkpoint.pt'
CHECKPOINT_MANIFEST_PATH = WEIGHTS_DIR / 'checkpoint_manifest.json'
for d in [CONFIG_DIR, WEIGHTS_DIR, TRAINING_LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)



print('Final training condition:')
print('  MICROSCOPE_LABEL =', MICROSCOPE_LABEL)
print('  SUPERVISION_TARGET =', SUPERVISION_TARGET)
print('  RESET_SAM2_DATASET =', RESET_SAM2_DATASET)
if bool(RESET_SAM2_DATASET):
    raise ValueError('E08 does not rebuild the E07-owned SAM2 VOS cache. Run supplemental/E07.ipynb with RESET_EXISTING=True if the E07 raw dataset and derived SAM2 cache must be rebuilt.')
print('  E08 consumes the E07-owned SAM2 VOS cache; weights/logs remain under E08.')
print('  RESET_TRAINING_CHECKPOINTS =', RESET_TRAINING_CHECKPOINTS)
print('  CHECKPOINT_RECOVERY_ONLY =', CHECKPOINT_RECOVERY_ONLY)
print('  DATASET_ONLY =', DATASET_ONLY)
print('  EMPTY_GOOGLE_DRIVE_TRASH_ON_CLEANUP =', EMPTY_GOOGLE_DRIVE_TRASH_ON_CLEANUP)
print('  SAM2_TRAIN_NUM_FRAMES =', SAM2_TRAIN_NUM_FRAMES)
print('  SAM2_MAX_NUM_OBJECTS =', SAM2_MAX_NUM_OBJECTS)
print('  SAM2_REVERSE_TIME_PROB =', SAM2_REVERSE_TIME_PROB)
print('  SAM2_VAL_FRACTION =', SAM2_VAL_FRACTION)
print('  SAM2_VAL_RANDOM_SEED =', SAM2_VAL_RANDOM_SEED)
print('  SAM2_LOSS_SEMANTICS_VERSION =', SAM2_LOSS_SEMANTICS_VERSION)
print('  SAM2_MIN_VALID_FRAMES =', SAM2_MIN_VALID_FRAMES)
print('  RAW_DATASET_ROOT =', RAW_DATASET_ROOT)
print('  FINAL_CHECKPOINT_PATH =', FINAL_CHECKPOINT_PATH)

# Authorize Google Drive API access early when checkpoint cleanup may empty Trash.
# drive.mount() and Drive API deletion use separate authorization paths in Colab.
drive_service = None
if bool(EMPTY_GOOGLE_DRIVE_TRASH_ON_CLEANUP):
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
        print('Authorizing Google Drive API for Trash cleanup now...')
        auth.authenticate_user()
        drive_service = build('drive', 'v3')
        print('  Drive API authorized for Trash cleanup.')
    except Exception as exc:
        print('  Drive API authorization failed:', repr(exc))
        print('  Trash cleanup will be skipped unless authorization succeeds later.')
        drive_service = None
else:
    print('Drive API Trash cleanup authorization skipped; EMPTY_GOOGLE_DRIVE_TRASH_ON_CLEANUP=False.')


## Runtime split and resume behavior

E07 generates the raw Syniscopy dataset and the E07-owned SAM2 VOS cache. E08 consumes that cache and trains or resumes the SAM2 checkpoint; it does not rebuild the cache.

- Run E07 first so `outputs/E07/dataset/raw_syniscopy/dataset_manifest.json` and `outputs/E07/dataset/sam2_vos/conversion_manifest.json` exist.
- Set `DATASET_ONLY = True` in E08 to verify the E07 raw dataset and SAM2 cache paths, then stop before SAM2 setup/training.
- Keep `RESET_SAM2_DATASET = False` in E08. To rebuild the E07-owned cache, rerun `supplemental/E07.ipynb` deliberately.
- Keep `RESET_TRAINING_CHECKPOINTS = False` to resume SAM2 training from `training_logs/checkpoints/checkpoint.pt`.
- Set `RESET_TRAINING_CHECKPOINTS = True` only when intentionally discarding prior E08 training for this label.
- If Colab disconnected after SAM2 saved a checkpoint and you want to move on without more training, set `CHECKPOINT_RECOVERY_ONLY = True`; the notebook exports the best/resume checkpoint to `weights/final_checkpoint.pt` and stops before launching training.


In [ ]:
# CELL 0.8: Dataset-only check for the E07 raw dataset and E07-owned SAM2 cache
# E07 owns raw Syniscopy dataset generation and the derived SAM2 VOS cache.
# E08 only consumes those assets while writing weights/logs under the E08 label.
if bool(globals().get('DATASET_ONLY', False)):
    raw_manifest_path = RAW_DATASET_ROOT / 'dataset_manifest.json'
    sam2_manifest_path = SAM2_DATASET_ROOT / 'conversion_manifest.json'
    frames_path = SAM2_DATASET_ROOT / 'Frames'
    gt_path = SAM2_DATASET_ROOT / 'GT'
    ignore_path = SAM2_DATASET_ROOT / 'Ignore'
    loss_weight_path = SAM2_DATASET_ROOT / 'LossWeight'
    train_list_path = SAM2_DATASET_ROOT / 'train_list.txt'
    val_list_path = SAM2_DATASET_ROOT / 'val_list.txt'
    print('DATASET_ONLY=True: checking E07 raw dataset and SAM2 cache only.')
    print('  RAW_DATASET_ROOT =', RAW_DATASET_ROOT)
    print('  SAM2_DATASET_ROOT =', SAM2_DATASET_ROOT)
    if not raw_manifest_path.exists():
        raise FileNotFoundError(f'E07 raw dataset is missing at {raw_manifest_path}. Run supplemental/E07.ipynb first.')
    required_cache_paths = [sam2_manifest_path, frames_path, gt_path, ignore_path, loss_weight_path, train_list_path, val_list_path]
    missing = [str(p) for p in required_cache_paths if not p.exists()]
    if missing:
        raise FileNotFoundError('E07 SAM2 VOS cache is missing. Run supplemental/E07.ipynb to build it. Missing: ' + ', '.join(missing))
    print('Dataset and SAM2 cache ready.')
    raise SystemExit('DATASET_ONLY complete')
else:
    print('DATASET_ONLY=False: continuing to SAM2 setup/training cells.')


In [ ]:
# CELL 1: Prepare SAM2 repo and install dependencies

import os
import shutil
import subprocess
import sys
from pathlib import Path

print("=" * 70)
print("  INITIAL SETUP: Clone SAM2 & Install Dependencies")
print("=" * 70)

SAM2_DIR = Path('/content/segment-anything-2')
SAM2_GIT_URL = 'https://github.com/facebookresearch/segment-anything-2.git'
SAM2_GIT_COMMIT = '2b90b9f5ceec907a1c18123530e92e794ad901a4'
os.chdir('/content')


def _current_git_commit(repo_dir: Path) -> str | None:
    try:
        out = subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'],
            cwd=str(repo_dir),
            stderr=subprocess.DEVNULL,
            text=True,
        )
    except Exception:
        return None
    return out.strip() or None


repo_valid = (
    (SAM2_DIR / 'setup.py').exists()
    and (SAM2_DIR / 'training' / 'train.py').exists()
    and (SAM2_DIR / 'sam2').is_dir()
    and _current_git_commit(SAM2_DIR) == SAM2_GIT_COMMIT
)

if repo_valid:
    print(f"✅ Reusing pinned SAM2 repository: {SAM2_DIR} @ {SAM2_GIT_COMMIT[:12]}")
else:
    if SAM2_DIR.exists():
        print("Removing SAM2 directory that is missing files or not at the pinned commit...")
        shutil.rmtree(SAM2_DIR)
        print("✅ Removed")

    print("\nCloning pinned SAM2 repository...")
    subprocess.run(['git', 'clone', SAM2_GIT_URL, str(SAM2_DIR)], check=True)
    subprocess.run(['git', 'checkout', '-q', SAM2_GIT_COMMIT], cwd=str(SAM2_DIR), check=True)

    if not (SAM2_DIR / 'setup.py').exists():
        raise RuntimeError("Clone failed: setup.py missing")
    if not (SAM2_DIR / 'training' / 'train.py').exists():
        raise RuntimeError("Clone failed: training/train.py missing")
    actual_commit = _current_git_commit(SAM2_DIR)
    if actual_commit != SAM2_GIT_COMMIT:
        raise RuntimeError(f"SAM2 checkout mismatch: {actual_commit} != {SAM2_GIT_COMMIT}")
    print(f"✅ Repository cloned successfully at {SAM2_GIT_COMMIT[:12]}")

os.chdir(SAM2_DIR)
print("\nInstalling SAM2 package...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print("✅ SAM2 installed")

print("\n" + "=" * 70)
print("  SAM2 SETUP COMPLETE — Proceed to checkpoint verification")
print("=" * 70)


In [ ]:
# CELL 1.5: Verify/download SAM 2.1 Large checkpoint

import os
from pathlib import Path

SAM2_DIR = Path('/content/segment-anything-2')
SAM2_BASE_CHECKPOINT_FILENAME = 'sam2.1_hiera_large.pt'
SAM2_BASE_CHECKPOINT_URL = 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt'
SAM2_CONFIG_NAME = 'configs/sam2.1/sam2.1_hiera_l.yaml'
CKPT_PATH = SAM2_DIR / SAM2_BASE_CHECKPOINT_FILENAME

print("=" * 70)
print("  CHECKPOINT VERIFICATION")
print("=" * 70)

needs_download = False
if CKPT_PATH.exists():
    size_mb = CKPT_PATH.stat().st_size / (1024 * 1024)
    print(f"📦 Checkpoint exists: {CKPT_PATH}")
    print(f"   Size: {size_mb:.1f} MB")
    if size_mb < 100:
        print("   File too small; replacing it.")
        needs_download = True
    else:
        print("   ✅ Size looks valid")
else:
    print(f"Checkpoint not found: {CKPT_PATH}")
    needs_download = True

if needs_download:
    print("\nDownloading SAM 2.1 Large checkpoint...")
    !wget -q --show-progress -O {CKPT_PATH} {SAM2_BASE_CHECKPOINT_URL}
    if not CKPT_PATH.exists() or CKPT_PATH.stat().st_size < 100 * 1024 * 1024:
        raise RuntimeError(f'Checkpoint download failed or produced an invalid file: {CKPT_PATH}')
    print(f"✅ Download complete: {CKPT_PATH} ({CKPT_PATH.stat().st_size / (1024 * 1024):.1f} MB)")

print('SAM2 config:', SAM2_CONFIG_NAME)
print("=" * 70)


In [ ]:
# CELL 2: Dataset helpers for Syniscopy supervision masks -> SAM2

import os
import cv2
from pathlib import Path
import numpy as np
import json
from collections import defaultdict

from dataset_schema import annotation_paths_for_frame, validate_supervision_target

SUPERVISION_TARGET = validate_supervision_target(SUPERVISION_TARGET)
print('SAM2 supervision target:', SUPERVISION_TARGET)

def load_dataset_manifest(dataset_root):
    manifest_path = os.path.join(dataset_root, 'dataset_manifest.json')
    if not os.path.exists(manifest_path):
        raise FileNotFoundError(f'dataset_manifest.json not found at: {manifest_path}')
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)
    if 'videos' not in manifest or not isinstance(manifest['videos'], list):
        raise ValueError("dataset_manifest.json must contain a 'videos' list")
    return manifest

def load_video_metadata(dataset_root, video_id):
    meta_path = os.path.join(dataset_root, 'metadata', f'{video_id}.json')
    if not os.path.exists(meta_path):
        raise FileNotFoundError(f'Metadata JSON not found for video_id={video_id} at: {meta_path}')
    with open(meta_path, 'r') as f:
        meta = json.load(f)
    if 'num_frames' not in meta:
        raise ValueError(f"Metadata for {video_id} missing 'num_frames' field")
    if 'particles' not in meta or not isinstance(meta['particles'], list):
        raise ValueError(f"Metadata for {video_id} must contain a 'particles' list")
    return meta

def resolve_frame_sequence_dir(dataset_root, vid_entry):
    video_id = vid_entry.get('video_id')
    frame_rel = vid_entry.get('training_frames_dir') or vid_entry.get('frame_sequence_dir')
    if video_id is None or frame_rel is None:
        raise ValueError(f'Malformed manifest entry missing video_id/frame_sequence_dir: {vid_entry}')
    frame_dir = os.path.join(dataset_root, frame_rel)
    if not os.path.isdir(frame_dir):
        raise FileNotFoundError(f'Lossless frame sequence missing for video_id={video_id}: {frame_dir}')
    return frame_dir

def load_frame_sequence(frame_dir):
    paths = sorted(Path(frame_dir).glob('*.png'))
    if not paths:
        raise RuntimeError(f'Frame sequence contains zero PNG frames: {frame_dir}')
    frames = []
    for path in paths:
        frame_bgr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        if frame_bgr is None:
            raise RuntimeError(f'Failed to read frame sequence image: {path}')
        if frame_bgr.ndim == 2:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_GRAY2RGB)
        elif frame_bgr.ndim == 3 and frame_bgr.shape[2] == 3:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        else:
            raise RuntimeError(f'Unsupported frame shape in {path}: {frame_bgr.shape}')
        frames.append(frame_rgb)
    return frames

def build_video_frame_index(dataset_root, allowed_video_ids=None):
    dataset_root = os.path.abspath(dataset_root)
    manifest = load_dataset_manifest(dataset_root)
    all_entries = []
    videos_info = manifest['videos']
    if allowed_video_ids is not None:
        allowed_video_ids = {str(v) for v in allowed_video_ids}
        videos_info = [v for v in videos_info if str(v.get('video_id')) in allowed_video_ids]
    print(f'Found {len(videos_info)} selected videos in dataset_manifest.json')
    total_target_masks = 0
    missing_target_masks = 0

    for vid_entry in videos_info:
        video_id = vid_entry.get('video_id')
        mask_root_rel = vid_entry.get('mask_root_dir')
        if video_id is None or mask_root_rel is None:
            print(f'Skipping malformed video entry in manifest: {vid_entry}')
            continue

        frame_dir = resolve_frame_sequence_dir(dataset_root, vid_entry)
        mask_root = os.path.join(dataset_root, mask_root_rel)
        schema_path = os.path.join(mask_root, 'annotation_schema.json')
        if not os.path.exists(schema_path):
            raise FileNotFoundError(f'Annotation schema missing for video_id={video_id}: {schema_path}')

        meta = load_video_metadata(dataset_root, video_id)
        num_frames_meta = int(meta['num_frames'])
        particles = meta['particles']
        print(f"\nProcessing video_id={video_id}")
        print(f'  Frame sequence: {frame_dir}')
        print(f'  Mask root: {mask_root}')
        print(f'  Declared num_frames: {num_frames_meta}')
        print(f'  Number of particles: {len(particles)}')

        frames = load_frame_sequence(frame_dir)
        num_frames_video = len(frames)
        if num_frames_video != num_frames_meta:
            print(f'  [Warning] Frame sequence {video_id}: found {num_frames_video} frames, metadata says {num_frames_meta}. Using sequence count.')

        for f_idx in range(num_frames_video):
            frame_img = frames[f_idx]
            target_mask_paths = []
            ignore_mask_paths = []
            loss_weight_paths = []
            for pinfo in particles:
                if 'particle_index' not in pinfo:
                    continue
                p_idx = int(pinfo['particle_index'])
                paths = annotation_paths_for_frame(
                    video_id=video_id,
                    mask_root=mask_root,
                    particle_index=p_idx,
                    frame_index=f_idx,
                    target=SUPERVISION_TARGET,
                )
                if paths.target_mask is not None:
                    target_mask_paths.append(str(paths.target_mask))
                    total_target_masks += 1
                else:
                    missing_target_masks += 1
                if paths.ignore_mask is not None:
                    ignore_mask_paths.append(str(paths.ignore_mask))
                if paths.loss_weight is not None:
                    loss_weight_paths.append(str(paths.loss_weight))
            all_entries.append({
                'video_id': video_id,
                'frame_index': f_idx,
                'image': frame_img,
                'mask_paths': target_mask_paths,
                'ignore_mask_paths': ignore_mask_paths,
                'loss_weight_paths': loss_weight_paths,
            })
        print(f'  Collected {num_frames_video} frame entries for video_id={video_id}')

    print(f"\nTotal frame entries across all videos: {len(all_entries)}")
    print(f'Target mask paths found: {total_target_masks}; missing target paths: {missing_target_masks}')
    if all_entries and total_target_masks == 0:
        raise RuntimeError(f'No {SUPERVISION_TARGET} target masks were found. Check mask_output_directory and annotation schema.')
    return all_entries

def group_entries_by_video(all_entries):
    videos_by_id = defaultdict(list)
    for e in all_entries:
        videos_by_id[e['video_id']].append(e)
    return {vid: sorted(entries, key=lambda x: x['frame_index']) for vid, entries in videos_by_id.items()}

def _read_binary_union(paths, shape_hw):
    h, w = shape_hw
    union = np.zeros((h, w), dtype=np.uint8)
    for path in paths or []:
        if not os.path.exists(path):
            continue
        m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if m.shape[:2] != (h, w):
            m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
        union = np.maximum(union, (m > 0).astype(np.uint8))
    return union

def _read_loss_weight_union(paths, shape_hw):
    h, w = shape_hw
    if not paths:
        return np.ones((h, w), dtype=np.float32)
    weight = np.zeros((h, w), dtype=np.float32)
    found = False
    for path in paths:
        if not os.path.exists(path):
            continue
        m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        found = True
        if m.shape[:2] != (h, w):
            m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
        weight = np.maximum(weight, m.astype(np.float32) / 255.0)
    return weight if found else np.ones((h, w), dtype=np.float32)

def build_union_maps_for_entry(entry):
    img = entry['image']
    h, w = img.shape[:2]
    target_mask = _read_binary_union(entry.get('mask_paths', []), (h, w))
    ignore_mask = _read_binary_union(entry.get('ignore_mask_paths', []), (h, w))
    loss_weight = _read_loss_weight_union(entry.get('loss_weight_paths', []), (h, w))
    loss_weight[ignore_mask > 0] = 0.0
    has_fg = np.count_nonzero(target_mask) > 0
    return target_mask, ignore_mask, loss_weight, has_fg

def build_union_mask_for_entry(entry):
    target_mask, _, _, has_fg = build_union_maps_for_entry(entry)
    return target_mask, has_fg

print('✅ Dataset helper functions loaded')



In [ ]:
# CELL 3: Use the E07 raw Syniscopy dataset inside Colab
manifest_path = RAW_DATASET_ROOT / 'dataset_manifest.json'
print('Raw dataset source:')
print('  MICROSCOPE_LABEL =', MICROSCOPE_LABEL)
print('  RAW_DATASET_ROOT =', RAW_DATASET_ROOT)
print('  SAM2_DATASET_ROOT =', SAM2_DATASET_ROOT)

if not manifest_path.exists():
    raise FileNotFoundError(f'E07 raw dataset manifest is missing at {manifest_path}. Run supplemental/E07.ipynb first.')
print('Raw dataset ready:', RAW_DATASET_ROOT)


In [ ]:
%cd /content/segment-anything-2

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import random

DATASET_ROOT = str(RAW_DATASET_ROOT)
if not os.path.exists(DATASET_ROOT):
    raise FileNotFoundError(f"Dataset not found: {DATASET_ROOT}")

print(f"Using Syniscopy dataset root: {DATASET_ROOT}")
_manifest = load_dataset_manifest(DATASET_ROOT)
_all_video_ids = sorted(str(v.get('video_id')) for v in _manifest['videos'] if v.get('video_id'))
if len(_all_video_ids) < 2:
    raise RuntimeError("Need at least two videos for a train/validation preview split")

_rng = random.Random(int(SAM2_VAL_RANDOM_SEED))
_shuffled_ids = list(_all_video_ids)
_rng.shuffle(_shuffled_ids)
_val_count = max(1, int(round(len(_shuffled_ids) * float(SAM2_VAL_FRACTION))))
_val_count = min(_val_count, len(_shuffled_ids) - 1)
preview_val_ids = set(_shuffled_ids[:_val_count])
preview_train_ids = set(_shuffled_ids[_val_count:])

# This cell is only a visual smoke check. Load one train and one validation
# video from the E07-owned SAM2 cache instead of scanning the full raw E07
# corpus. The E08 validation cells already checked cache completeness.
_preview_ids = []
if preview_train_ids:
    _preview_ids.append(sorted(preview_train_ids)[0])
if preview_val_ids:
    _preview_ids.append(sorted(preview_val_ids)[0])

all_entries = build_video_frame_index(DATASET_ROOT, allowed_video_ids=_preview_ids)
if len(all_entries) == 0:
    raise RuntimeError("No preview frame entries found")

videos_by_id = group_entries_by_video(all_entries)
print(f"Preview videos loaded: {len(videos_by_id)} of {len(_all_video_ids)} total videos")

train_data = [e for e in all_entries if e["video_id"] in preview_train_ids]
test_data = [e for e in all_entries if e["video_id"] in preview_val_ids]

print(f"\nTraining split videos: {len(preview_train_ids)} total; preview samples loaded: {len(train_data)}")
print(f"Validation split videos: {len(preview_val_ids)} total; preview samples loaded: {len(test_data)}")

# SAM2 imports are loaded only in the runtime cells that need them
def read_batch(data, visualize_data=False):
    if len(data) == 0:
        print("Error: Empty data list")
        return None, None, None, 0

    ent = data[np.random.randint(len(data))]
    Img = ent["image"]
    if Img is None:
        return None, None, None, 0

    h, w = Img.shape[:2]
    ann_map, ignore_map, loss_weight_map, _ = build_union_maps_for_entry(ent)

    if np.count_nonzero(ann_map) == 0:
        if visualize_data:
            print("No labels in this frame")
        return Img, np.zeros((1, h, w), dtype=np.uint8), None, 0

    r = min(1024 / w, 1024 / h)
    new_w = int(w * r)
    new_h = int(h * r)
    Img = cv2.resize(Img, (new_w, new_h), interpolation=cv2.INTER_CUBIC)
    ann_map = cv2.resize(ann_map, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
    ignore_map = cv2.resize(ignore_map, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
    loss_weight_map = cv2.resize(loss_weight_map, (new_w, new_h), interpolation=cv2.INTER_NEAREST)

    binary_mask = (ann_map > 0).astype(np.uint8)
    eroded = cv2.erode(binary_mask, np.ones((5, 5), np.uint8), iterations=1)
    coords = np.argwhere(eroded > 0)

    points = []
    if len(coords) > 0:
        yx = coords[np.random.randint(len(coords))]
        points.append([yx[1], yx[0]])

    points = np.array(points, dtype=np.int32)

    if visualize_data:
        plt.figure(figsize=(15, 5))
        plt.subplot(1, 3, 1)
        plt.title('Frame')
        plt.imshow(Img)
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.title('Mask / Ignore overlay')
        overlay = np.stack([binary_mask, ignore_map > 0, np.zeros_like(binary_mask)], axis=-1).astype(float)
        plt.imshow(overlay)
        plt.axis('off')

        plt.subplot(1, 3, 3)
        plt.title('Prompt Points')
        plt.imshow(binary_mask, cmap='gray')
        if len(points) > 0:
            plt.scatter(points[0, 0], points[0, 1], c='red', s=100)
        plt.axis('off')
        plt.tight_layout()
        plt.show()

    binary_mask = np.expand_dims(binary_mask, axis=-1).transpose((2, 0, 1))
    points_out = np.expand_dims(points, axis=1) if points.size > 0 else None

    return Img, binary_mask, points_out, 1 if points_out is not None else 0

print("Testing data loader on a small preview subset...")
Img1, masks1, points1, num_masks = read_batch(train_data or test_data, visualize_data=True)
if Img1 is not None:
    print(f"✅ Image shape: {Img1.shape}")
    print(f"✅ Mask shape: {masks1.shape}")


In [ ]:
# CELL 4.5: Inspect the E07-owned SAM2 video-eligibility manifest

import json
from pathlib import Path

CANONICAL_TRAINING_FILTER_PATH = SAM2_DATASET_ROOT / 'sam2_training_filter.json'
LEGACY_TRAINING_FILTER_PATH = RAW_DATASET_ROOT / 'sam2_training_filter.json'
TRAINING_FILTER_PATH = CANONICAL_TRAINING_FILTER_PATH
CONVERSION_MANIFEST_PATH = SAM2_DATASET_ROOT / 'conversion_manifest.json'
print('=' * 70)
print('  E07-OWNED SAM2 CACHE MANIFESTS')
print('=' * 70)
print('  SAM2_DATASET_ROOT =', SAM2_DATASET_ROOT)
print('  canonical training filter =', CANONICAL_TRAINING_FILTER_PATH)
print('  legacy raw-root filter =', LEGACY_TRAINING_FILTER_PATH)
print('  conversion manifest =', CONVERSION_MANIFEST_PATH)

if not CONVERSION_MANIFEST_PATH.exists():
    raise FileNotFoundError(f'E07 SAM2 conversion manifest is missing at {CONVERSION_MANIFEST_PATH}. Run supplemental/E07.ipynb first.')

with open(CONVERSION_MANIFEST_PATH, 'r') as f:
    conversion_doc = json.load(f)

if CANONICAL_TRAINING_FILTER_PATH.exists():
    TRAINING_FILTER_PATH = CANONICAL_TRAINING_FILTER_PATH
elif LEGACY_TRAINING_FILTER_PATH.exists():
    TRAINING_FILTER_PATH = LEGACY_TRAINING_FILTER_PATH
else:
    TRAINING_FILTER_PATH = None

if TRAINING_FILTER_PATH is not None:
    with open(TRAINING_FILTER_PATH, 'r') as f:
        filter_doc = json.load(f)
    print('  Training filter used:', TRAINING_FILTER_PATH)
else:
    filter_doc = {
        'eligible_videos': [{'video_id': vid} for vid in conversion_doc.get('videos_kept', [])],
        'rejected_videos': conversion_doc.get('rejected', []),
    }
    print('  Training filter missing; using conversion_manifest.json as the authoritative E07 cache record.')

eligible_videos = filter_doc.get('eligible_videos', [])
rejected_videos = filter_doc.get('rejected_videos', [])
print('  Eligible videos:', len(eligible_videos))
print('  Rejected videos:', len(rejected_videos))
print('  Converted videos:', len(conversion_doc.get('videos_kept', [])))
print('  Frames kept:', conversion_doc.get('frames_kept'))
print('  Train videos:', len(conversion_doc.get('train_videos', [])))
print('  Val videos:', len(conversion_doc.get('val_videos', [])))
if not conversion_doc.get('videos_kept') or not conversion_doc.get('train_videos') or not conversion_doc.get('val_videos'):
    raise RuntimeError('E07 SAM2 cache has no converted/train/validation videos. Rebuild E07 SAM2 VOS cache.')


In [ ]:
# CELL 5: Validate E07-owned SAM2-format dataset

import json
from pathlib import Path

FRAMES_DIR = SAM2_DATASET_ROOT / 'Frames'
GT_DIR = SAM2_DATASET_ROOT / 'GT'
IGNORE_DIR = SAM2_DATASET_ROOT / 'Ignore'
LOSS_WEIGHT_DIR = SAM2_DATASET_ROOT / 'LossWeight'
LIST_PATH = SAM2_DATASET_ROOT / 'train_list.txt'
VAL_LIST_PATH = SAM2_DATASET_ROOT / 'val_list.txt'
CONVERSION_MANIFEST_PATH = SAM2_DATASET_ROOT / 'conversion_manifest.json'

required_paths = [FRAMES_DIR, GT_DIR, IGNORE_DIR, LOSS_WEIGHT_DIR, LIST_PATH, VAL_LIST_PATH, CONVERSION_MANIFEST_PATH]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError('E07 SAM2 VOS cache is incomplete. Run supplemental/E07.ipynb first. Missing: ' + ', '.join(missing))

with open(CONVERSION_MANIFEST_PATH, 'r') as f:
    conversion_manifest = json.load(f)

problems = []
converted_frames = 0
for vid in conversion_manifest.get('videos_kept', []):
    frame_names = {p.stem for p in (FRAMES_DIR / vid).glob('*.jpg')}
    gt_names = {p.stem for p in (GT_DIR / vid).glob('*.png')}
    ignore_names = {p.stem for p in (IGNORE_DIR / vid).glob('*.png')}
    loss_weight_names = {p.stem for p in (LOSS_WEIGHT_DIR / vid).glob('*.png')}
    converted_frames += len(frame_names)
    if not frame_names or not (frame_names == gt_names == ignore_names == loss_weight_names):
        problems.append({
            'video_id': vid,
            'frames': len(frame_names),
            'gt': len(gt_names),
            'ignore': len(ignore_names),
            'loss_weight': len(loss_weight_names),
        })
expected_frames = int(conversion_manifest.get('frames_kept', 0) or 0)
if expected_frames and converted_frames != expected_frames:
    problems.append({'reason': 'frame count mismatch', 'expected': expected_frames, 'actual': converted_frames})
if problems:
    raise RuntimeError('E07 SAM2 VOS cache failed validation: ' + json.dumps(problems[:10], indent=2))

print('✅ E07 SAM2 VOS cache is ready.')
print('   SAM2 dataset:', SAM2_DATASET_ROOT)
print('   Converted frames:', converted_frames)
print('   Training videos:', len(conversion_manifest.get('train_videos', [])))
print('   Validation videos:', len(conversion_manifest.get('val_videos', [])))
print('   Resize mode:', conversion_manifest.get('conversion_config', {}).get('resize_long_side_px', 'unknown'))


In [ ]:
# CELL 6: Install SAM2 training dependencies and restore the official SAM2 training source

%cd /content/segment-anything-2

import os
import subprocess
from pathlib import Path

print("=" * 70)
print("  INSTALLING TRAINING DEPENDENCIES")
print("=" * 70)

print("\nInstalling training dependencies...")
!pip install -q submitit hydra-core omegaconf fvcore iopath tqdm tensorboard
!pip install -q -e ".[dev]"
print("✅ Dependencies installed")

RESTORE_SAM2_TRAINING_FILES = [
    "training/train.py",
    "training/utils/data_utils.py",
    "training/dataset/vos_segment_loader.py",
    "training/dataset/vos_dataset.py",
    "training/dataset/transforms.py",
    "training/trainer.py",
    "training/loss_fns.py",
]

for rel_path in RESTORE_SAM2_TRAINING_FILES:
    if not Path(rel_path).exists():
        raise FileNotFoundError(f"{rel_path} missing; rerun the SAM2 clone/setup cell.")

# Start every training attempt from official SAM2 source files. Syniscopy custom
# behavior is installed in the training support step below.
subprocess.run(["git", "checkout", "-f", "--", *RESTORE_SAM2_TRAINING_FILES], check=True)
print("✅ Official SAM2 training source restored")
print("=" * 70)


In [ ]:
# CELL 6.5: CREATE CONSECUTIVE FRAME SAMPLER (COMPLETE - SAM2 COMPATIBLE)

import os

SAMPLER_FILE = "/content/segment-anything-2/training/dataset/consecutive_sampler.py"

print("=" * 70)
print("  CREATING CONSECUTIVE FRAME SAMPLER")
print("=" * 70)

sampler_code = '''"""
Consecutive Frame Sampler for SAM2 Video Segmentation
Matches SAM2's VOSSampler interface exactly
"""

import random
from training.dataset.vos_sampler import VOSSampler, SampledFramesAndObjects

MAX_RETRIES = 100


class ConsecutiveFrameSampler(VOSSampler):
    """
    Sample consecutive frames for video object segmentation.
    Drop-in replacement for RandomUniformSampler.
    """

    def __init__(self, num_frames, max_num_objects, max_gap=2, reverse_time_prob=0.0):
        self.num_frames = num_frames
        self.max_num_objects = max_num_objects
        self.max_gap = max_gap
        self.reverse_time_prob = reverse_time_prob

    def sample(self, video, segment_loader, epoch=None):
        """
        Sample consecutive frames from video.

        Args:
            video: Video object with .frames list
            segment_loader: Loader for segmentation masks
            epoch: Current training epoch (unused)

        Returns:
            SampledFramesAndObjects
        """

        for retry in range(MAX_RETRIES):
            if len(video.frames) < self.num_frames:
                raise Exception(
                    f"Cannot sample {self.num_frames} frames from video "
                    f"{video.video_name} (only has {len(video.frames)} frames)"
                )

            # Find consecutive frames
            frames = self._find_consecutive_frames(video.frames)

            # Optionally reverse time
            if random.uniform(0, 1) < self.reverse_time_prob:
                frames = frames[::-1]

            # Get visible objects in first frame
            visible_object_ids = []
            loaded_segms = segment_loader.load(frames[0].frame_idx)

            # Handle both dict and LazySegments
            if hasattr(loaded_segms, 'keys'):
                for object_id in loaded_segms.keys():
                    segment = loaded_segms[object_id]
                    if isinstance(segment, dict):
                        segment = segment.get('mask')
                    if hasattr(segment, 'sum'):
                        if segment.sum() > 0:
                            visible_object_ids.append(object_id)
                    else:
                        visible_object_ids.append(object_id)

            # First frame needs at least one object
            if len(visible_object_ids) > 0:
                break

            if retry >= MAX_RETRIES - 1:
                raise Exception("No visible objects in sampled frames")

        # Sample objects
        object_ids = random.sample(
            visible_object_ids,
            min(len(visible_object_ids), self.max_num_objects),
        )

        return SampledFramesAndObjects(frames=frames, object_ids=object_ids)

    def _find_consecutive_frames(self, all_frames):
        """Find num_frames that are consecutive or near-consecutive"""

        if len(all_frames) == self.num_frames:
            return all_frames

        # Try to find frames within max_gap
        for attempt in range(50):
            start_idx = random.randint(0, len(all_frames) - self.num_frames)

            selected = [all_frames[start_idx]]
            current_frame_idx = all_frames[start_idx].frame_idx

            for i in range(1, self.num_frames):
                found = False
                for j in range(start_idx + 1, min(start_idx + self.max_gap + 2, len(all_frames))):
                    candidate = all_frames[j]
                    if candidate.frame_idx - current_frame_idx <= self.max_gap + 1:
                        selected.append(candidate)
                        current_frame_idx = candidate.frame_idx
                        start_idx = j
                        found = True
                        break

                if not found:
                    break

            if len(selected) == self.num_frames:
                return selected

        # Fallback: consecutive from list
        start = random.randint(0, len(all_frames) - self.num_frames)
        return all_frames[start:start + self.num_frames]
'''

with open(SAMPLER_FILE, "w") as f:
    f.write(sampler_code)

print(f"✅ Created: {SAMPLER_FILE}")
print("=" * 70)


In [ ]:
# Validate the SAM2 pieces used by Syniscopy training.
# Stock framework pieces should stay stock; Syniscopy-specific behavior is checked explicitly.

import os
from pathlib import Path

SAM2_DIR = Path('/content/segment-anything-2')
SYNISCOPY_CONFIG_PATH = SAM2_DIR / 'sam2' / 'configs' / 'sam2.1_training' / 'sam2.1_hiera_l_syniscopy_finetune.yaml'
SAMPLER_FILE = SAM2_DIR / 'training' / 'dataset' / 'consecutive_sampler.py'
LOSS_FILE = SAM2_DIR / 'training' / 'loss_fns.py'
TRAIN_FILE = SAM2_DIR / 'training' / 'train.py'
DATA_ROOT = Path(SAM2_DATASET_ROOT)
LIST_PATH = DATA_ROOT / 'train_list.txt'

print('=' * 70)
print('  SAM2 training environment validation')
print('=' * 70)

issues_found = []

print('\n[1/5] Checking stock SAM2 training launcher...')
if TRAIN_FILE.exists():
    train_code = TRAIN_FILE.read_text()
    if 'initialize_config_module("sam2", version_base="1.2")' in train_code:
        print('   ✅ training/train.py uses the official SAM2 config module')
    else:
        print('   ⚠️ training/train.py does not look like the stock SAM2 launcher')
        issues_found.append('TRAIN_LAUNCHER_NOT_STOCK')
else:
    print('   ❌ training/train.py not found')
    issues_found.append('TRAIN_FILE_MISSING')

print('\n[2/5] Checking Syniscopy consecutive-frame sampler...')
if SAMPLER_FILE.exists():
    sampler_code = SAMPLER_FILE.read_text()
    if 'class ConsecutiveFrameSampler' in sampler_code and 'def sample(' in sampler_code:
        print('   ✅ Syniscopy consecutive sampler is installed')
    else:
        print('   ❌ ConsecutiveFrameSampler is incomplete')
        issues_found.append('SYNISCOPY_SAMPLER_INCOMPLETE')
else:
    print('   ❌ Syniscopy consecutive sampler not found')
    issues_found.append('SYNISCOPY_SAMPLER_MISSING')

print('\n[3/5] Checking official SAM2 Hydra config location...')
if SYNISCOPY_CONFIG_PATH.exists():
    cfg = SYNISCOPY_CONFIG_PATH.read_text()
    if 'training.dataset.consecutive_sampler.ConsecutiveFrameSampler' in cfg or 'training.dataset.consecutive_sampler' in cfg:
        print('   ✅ YAML uses Syniscopy consecutive sampler through Hydra')
    elif 'training.dataset.vos_sampler.RandomUniformSampler' in cfg:
        print('   ⚠️ YAML uses stock RandomUniformSampler')
    else:
        print('   ⚠️ Could not identify sampler target in YAML')
else:
    print('   ℹ️ Config YAML will be created in the training setup cell')

print('\n[4/5] Checking Syniscopy custom loss support state...')
if LOSS_FILE.exists():
    loss_code = LOSS_FILE.read_text()
    if 'Syniscopy supervision support' in loss_code and 'pixel_weight' in loss_code:
        print('   ✅ Ignore/LossWeight-aware loss support is installed')
    else:
        print('   ℹ️ Stock SAM2 loss is present; Syniscopy loss support installs in the training setup cell')
else:
    print('   ❌ loss_fns.py not found')
    issues_found.append('LOSS_FILE_MISSING')

print('\n[5/5] Checking converted dataset index...')
if LIST_PATH.exists():
    videos = [line.strip() for line in LIST_PATH.read_text().splitlines() if line.strip()]
    print(f'   ✅ Found {len(videos)} videos in train_list.txt')
    if len(videos) < 2:
        print(f'   ⚠️ Only {len(videos)} videos found')
        issues_found.append('FEW_VIDEOS')
else:
    print('   ℹ️ train_list.txt will be created by the dataset conversion cell')

print('\n' + '=' * 70)
print('  DIAGNOSTIC SUMMARY')
print('=' * 70)
if issues_found:
    print(f'⚠️ Found {len(issues_found)} issue(s):')
    for issue in issues_found:
        print(f'   - {issue}')
else:
    print('✅ No blocking issues found in the current setup state')
print('=' * 70)

print('\n' + '=' * 70)
print('  VERIFICATION: importing Syniscopy sampler')
print('=' * 70)
try:
    from training.dataset.consecutive_sampler import ConsecutiveFrameSampler
    sampler = ConsecutiveFrameSampler(num_frames=SAM2_TRAIN_NUM_FRAMES, max_num_objects=SAM2_MAX_NUM_OBJECTS, max_gap=2, reverse_time_prob=SAM2_REVERSE_TIME_PROB)
    print('✅ ConsecutiveFrameSampler imported successfully')
    print(f'   - num_frames: {sampler.num_frames}')
    print(f'   - max_num_objects: {sampler.max_num_objects}')
    print(f'   - max_gap: {sampler.max_gap}')
    print(f'   - reverse_time_prob: {sampler.reverse_time_prob}')
except Exception as e:
    print(f'❌ Error importing Syniscopy sampler: {e}')
    raise
print('=' * 70)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# COMBINED CELL 7 - SAM2 FINE-TUNING
# ═══════════════════════════════════════════════════════════════════════════════

import os
import re
import shutil
import time
import threading
import textwrap
import glob
import yaml
import subprocess
from pathlib import Path

print("=" * 80)
print("  COMBINED SAM2 FINE-TUNING SETUP")
print("=" * 80)

SAM2_TRAIN_NUM_FRAMES = int(SAM2_TRAIN_NUM_FRAMES)
SAM2_MAX_NUM_OBJECTS = int(SAM2_MAX_NUM_OBJECTS)
SAM2_REVERSE_TIME_PROB = float(SAM2_REVERSE_TIME_PROB)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1/8: Write YAML Config
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("STEP 1/8: Creating YAML Configuration")
print("=" * 80)

SAM2_CONFIG_RELATIVE = "configs/sam2.1_training/sam2.1_hiera_l_syniscopy_finetune.yaml"
SAM2_CONFIG_DIR = str(SAM2_DIR / "sam2" / "configs" / "sam2.1_training")
os.makedirs(SAM2_CONFIG_DIR, exist_ok=True)
CFG_PATH = os.path.join(SAM2_CONFIG_DIR, "sam2.1_hiera_l_syniscopy_finetune.yaml")

# Write YAML to file directly to avoid any triple-quote issues
yaml_lines = []
yaml_lines.append("# " + "@package _global_")
yaml_lines.append("")
yaml_lines.append("scratch:")
yaml_lines.append("  resolution: 1024")
yaml_lines.append("  train_batch_size: 1")
yaml_lines.append("  num_train_workers: 4")
yaml_lines.append(f"  num_frames: {int(SAM2_TRAIN_NUM_FRAMES)}")
yaml_lines.append(f"  max_num_objects: {int(SAM2_MAX_NUM_OBJECTS)}")
yaml_lines.append("  base_lr: 5.0e-6")
yaml_lines.append("  vision_lr: 3.0e-06")
yaml_lines.append("  phases_per_epoch: 1")
yaml_lines.append("  num_epochs: 40")
yaml_lines.append("")
yaml_lines.append("dataset:")
yaml_lines.append(f"  img_folder: {SAM2_DATASET_ROOT}/Frames")
yaml_lines.append(f"  gt_folder: {SAM2_DATASET_ROOT}/GT")
yaml_lines.append(f"  file_list_txt: {SAM2_DATASET_ROOT}/train_list.txt")
yaml_lines.append(f"  val_file_list_txt: {SAM2_DATASET_ROOT}/val_list.txt")
yaml_lines.append("  multiplier: 20")
yaml_lines.append("  val_multiplier: 1")
yaml_lines.append("")
yaml_lines.append("vos:")
yaml_lines.append("  train_transforms:")
yaml_lines.append("    - _target_: training.dataset.transforms.ComposeAPI")
yaml_lines.append("      transforms:")
yaml_lines.append("        - _target_: training.dataset.transforms.RandomHorizontalFlip")
yaml_lines.append("          consistent_transform: True")
yaml_lines.append("        - _target_: training.dataset.transforms.RandomAffine")
yaml_lines.append("          degrees: 25")
yaml_lines.append("          shear: 20")
yaml_lines.append("          image_interpolation: bilinear")
yaml_lines.append("          consistent_transform: True")
yaml_lines.append("        - _target_: training.dataset.transforms.RandomResizeAPI")
yaml_lines.append("          sizes: ${scratch.resolution}")
yaml_lines.append("          square: true")
yaml_lines.append("          consistent_transform: True")
yaml_lines.append("        - _target_: training.dataset.transforms.ColorJitter")
yaml_lines.append("          consistent_transform: True")
yaml_lines.append("          brightness: 0.1")
yaml_lines.append("          contrast: 0.03")
yaml_lines.append("          saturation: 0.03")
yaml_lines.append("          hue: null")
yaml_lines.append("        - _target_: training.dataset.transforms.RandomGrayscale")
yaml_lines.append("          p: 0.05")
yaml_lines.append("          consistent_transform: True")
yaml_lines.append("        - _target_: training.dataset.transforms.ColorJitter")
yaml_lines.append("          consistent_transform: False")
yaml_lines.append("          brightness: 0.1")
yaml_lines.append("          contrast: 0.05")
yaml_lines.append("          saturation: 0.05")
yaml_lines.append("          hue: null")
yaml_lines.append("        - _target_: training.dataset.transforms.ToTensorAPI")
yaml_lines.append("        - _target_: training.dataset.transforms.NormalizeAPI")
yaml_lines.append("          mean: [0.485, 0.456, 0.406]")
yaml_lines.append("          std: [0.229, 0.224, 0.225]")
yaml_lines.append("  val_transforms:")
yaml_lines.append("    - _target_: training.dataset.transforms.ComposeAPI")
yaml_lines.append("      transforms:")
yaml_lines.append("        - _target_: training.dataset.transforms.RandomResizeAPI")
yaml_lines.append("          sizes: ${scratch.resolution}")
yaml_lines.append("          square: true")
yaml_lines.append("          consistent_transform: True")
yaml_lines.append("        - _target_: training.dataset.transforms.ToTensorAPI")
yaml_lines.append("        - _target_: training.dataset.transforms.NormalizeAPI")
yaml_lines.append("          mean: [0.485, 0.456, 0.406]")
yaml_lines.append("          std: [0.229, 0.224, 0.225]")
yaml_lines.append("")
yaml_lines.append("trainer:")
yaml_lines.append("  _target_: training.trainer.Trainer")
yaml_lines.append("  mode: train")
yaml_lines.append("  max_epochs: ${times:${scratch.num_epochs},${scratch.phases_per_epoch}}")
yaml_lines.append("  accelerator: cuda")
yaml_lines.append("  seed_value: 123")
yaml_lines.append("  val_epoch_freq: 1")
yaml_lines.append("")
yaml_lines.append("  model:")
yaml_lines.append("    _target_: training.model.sam2.SAM2Train")
yaml_lines.append("    image_encoder:")
yaml_lines.append("      _target_: sam2.modeling.backbones.image_encoder.ImageEncoder")
yaml_lines.append("      scalp: 1")
yaml_lines.append("      trunk:")
yaml_lines.append("        _target_: sam2.modeling.backbones.hieradet.Hiera")
yaml_lines.append("        embed_dim: 144")
yaml_lines.append("        num_heads: 2")
yaml_lines.append("        stages: [2, 6, 36, 4]")
yaml_lines.append("        global_att_blocks: [23, 33, 43]")
yaml_lines.append("        window_pos_embed_bkg_spatial_size: [7, 7]")
yaml_lines.append("        window_spec: [8, 4, 16, 8]")
yaml_lines.append("      neck:")
yaml_lines.append("        _target_: sam2.modeling.backbones.image_encoder.FpnNeck")
yaml_lines.append("        position_encoding:")
yaml_lines.append("          _target_: sam2.modeling.position_encoding.PositionEmbeddingSine")
yaml_lines.append("          num_pos_feats: 256")
yaml_lines.append("          normalize: true")
yaml_lines.append("          scale: null")
yaml_lines.append("          temperature: 10000")
yaml_lines.append("        d_model: 256")
yaml_lines.append("        backbone_channel_list: [1152, 576, 288, 144]")
yaml_lines.append("        fpn_top_down_levels: [2, 3]")
yaml_lines.append("        fpn_interp_model: nearest")
yaml_lines.append("")
yaml_lines.append("    memory_attention:")
yaml_lines.append("      _target_: sam2.modeling.memory_attention.MemoryAttention")
yaml_lines.append("      d_model: 256")
yaml_lines.append("      pos_enc_at_input: true")
yaml_lines.append("      layer:")
yaml_lines.append("        _target_: sam2.modeling.memory_attention.MemoryAttentionLayer")
yaml_lines.append("        activation: relu")
yaml_lines.append("        dim_feedforward: 2048")
yaml_lines.append("        dropout: 0.1")
yaml_lines.append("        pos_enc_at_attn: false")
yaml_lines.append("        self_attention:")
yaml_lines.append("          _target_: sam2.modeling.sam.transformer.RoPEAttention")
yaml_lines.append("          rope_theta: 10000.0")
yaml_lines.append("          feat_sizes: [64, 64]")
yaml_lines.append("          embedding_dim: 256")
yaml_lines.append("          num_heads: 1")
yaml_lines.append("          downsample_rate: 1")
yaml_lines.append("          dropout: 0.1")
yaml_lines.append("        d_model: 256")
yaml_lines.append("        pos_enc_at_cross_attn_keys: true")
yaml_lines.append("        pos_enc_at_cross_attn_queries: false")
yaml_lines.append("        cross_attention:")
yaml_lines.append("          _target_: sam2.modeling.sam.transformer.RoPEAttention")
yaml_lines.append("          rope_theta: 10000.0")
yaml_lines.append("          feat_sizes: [64, 64]")
yaml_lines.append("          rope_k_repeat: True")
yaml_lines.append("          embedding_dim: 256")
yaml_lines.append("          num_heads: 1")
yaml_lines.append("          downsample_rate: 1")
yaml_lines.append("          dropout: 0.1")
yaml_lines.append("          kv_in_dim: 64")
yaml_lines.append("      num_layers: 4")
yaml_lines.append("")
yaml_lines.append("    memory_encoder:")
yaml_lines.append("      _target_: sam2.modeling.memory_encoder.MemoryEncoder")
yaml_lines.append("      out_dim: 64")
yaml_lines.append("      position_encoding:")
yaml_lines.append("        _target_: sam2.modeling.position_encoding.PositionEmbeddingSine")
yaml_lines.append("        num_pos_feats: 64")
yaml_lines.append("        normalize: true")
yaml_lines.append("        scale: null")
yaml_lines.append("        temperature: 10000")
yaml_lines.append("      mask_downsampler:")
yaml_lines.append("        _target_: sam2.modeling.memory_encoder.MaskDownSampler")
yaml_lines.append("        kernel_size: 3")
yaml_lines.append("        stride: 2")
yaml_lines.append("        padding: 1")
yaml_lines.append("      fuser:")
yaml_lines.append("        _target_: sam2.modeling.memory_encoder.Fuser")
yaml_lines.append("        layer:")
yaml_lines.append("          _target_: sam2.modeling.memory_encoder.CXBlock")
yaml_lines.append("          dim: 256")
yaml_lines.append("          kernel_size: 7")
yaml_lines.append("          padding: 3")
yaml_lines.append("          layer_scale_init_value: 1e-6")
yaml_lines.append("          use_dwconv: True")
yaml_lines.append("        num_layers: 2")
yaml_lines.append("")
yaml_lines.append("    num_maskmem: 7")
yaml_lines.append("    image_size: ${scratch.resolution}")
yaml_lines.append("    sigmoid_scale_for_mem_enc: 20.0")
yaml_lines.append("    sigmoid_bias_for_mem_enc: -10.0")
yaml_lines.append("    use_mask_input_as_output_without_sam: true")
yaml_lines.append("    directly_add_no_mem_embed: true")
yaml_lines.append("    no_obj_embed_spatial: true")
yaml_lines.append("    use_high_res_features_in_sam: true")
yaml_lines.append("    multimask_output_in_sam: true")
yaml_lines.append("    iou_prediction_use_sigmoid: True")
yaml_lines.append("    use_obj_ptrs_in_encoder: true")
yaml_lines.append("    add_tpos_enc_to_obj_ptrs: true")
yaml_lines.append("    proj_tpos_enc_in_obj_ptrs: true")
yaml_lines.append("    use_signed_tpos_enc_to_obj_ptrs: true")
yaml_lines.append("    only_obj_ptrs_in_the_past_for_eval: true")
yaml_lines.append("    pred_obj_scores: true")
yaml_lines.append("    pred_obj_scores_mlp: true")
yaml_lines.append("    fixed_no_obj_ptr: true")
yaml_lines.append("    multimask_output_for_tracking: true")
yaml_lines.append("    use_multimask_token_for_obj_ptr: true")
yaml_lines.append("    multimask_min_pt_num: 0")
yaml_lines.append("    multimask_max_pt_num: 1")
yaml_lines.append("    use_mlp_for_obj_ptr_proj: true")
yaml_lines.append("")
yaml_lines.append("    prob_to_use_pt_input_for_train: 1.0")
yaml_lines.append("    prob_to_use_pt_input_for_eval: 1.0")
yaml_lines.append("    prob_to_use_box_input_for_train: 0.0")
yaml_lines.append("    prob_to_use_box_input_for_eval: 0.0")
yaml_lines.append("    prob_to_sample_from_gt_for_train: 0.1")
yaml_lines.append("    num_frames_to_correct_for_train: 2")
yaml_lines.append("    num_frames_to_correct_for_eval: 1")
yaml_lines.append("    rand_frames_to_correct_for_train: True")
yaml_lines.append("    add_all_frames_to_correct_as_cond: True")
yaml_lines.append("    num_init_cond_frames_for_train: 1")
yaml_lines.append("    rand_init_cond_frames_for_train: True")
yaml_lines.append("    num_correction_pt_per_frame: 7")
yaml_lines.append("    use_act_ckpt_iterative_pt_sampling: false")
yaml_lines.append("    num_init_cond_frames_for_eval: 1")
yaml_lines.append("    forward_backbone_per_frame_for_eval: True")
yaml_lines.append("")
yaml_lines.append("  data:")
yaml_lines.append("    train:")
yaml_lines.append("      _target_: training.dataset.sam2_datasets.TorchTrainMixedDataset")
yaml_lines.append("      phases_per_epoch: ${scratch.phases_per_epoch}")
yaml_lines.append("      batch_sizes:")
yaml_lines.append("        - ${scratch.train_batch_size}")
yaml_lines.append("      datasets:")
yaml_lines.append("        - _target_: training.dataset.utils.RepeatFactorWrapper")
yaml_lines.append("          dataset:")
yaml_lines.append("            _target_: training.dataset.utils.ConcatDataset")
yaml_lines.append("            datasets:")
yaml_lines.append("            - _target_: training.dataset.vos_dataset.VOSDataset")
yaml_lines.append("              transforms: ${vos.train_transforms}")
yaml_lines.append("              training: true")
yaml_lines.append("              video_dataset:")
yaml_lines.append("                _target_: training.dataset.vos_raw_dataset.PNGRawDataset")
yaml_lines.append("                img_folder: ${dataset.img_folder}")
yaml_lines.append("                gt_folder: ${dataset.gt_folder}")
yaml_lines.append("                file_list_txt: ${dataset.file_list_txt}")
yaml_lines.append("              sampler:")
yaml_lines.append("                _target_: training.dataset.consecutive_sampler.ConsecutiveFrameSampler")
yaml_lines.append("                num_frames: ${scratch.num_frames}")
yaml_lines.append("                max_num_objects: ${scratch.max_num_objects}")
yaml_lines.append("                max_gap: 2")
yaml_lines.append(f"                reverse_time_prob: {float(SAM2_REVERSE_TIME_PROB)}")
yaml_lines.append("              multiplier: ${dataset.multiplier}")
yaml_lines.append("      shuffle: True")
yaml_lines.append("      num_workers: ${scratch.num_train_workers}")
yaml_lines.append("      pin_memory: True")
yaml_lines.append("      drop_last: True")
yaml_lines.append("      collate_fn:")
yaml_lines.append("        _target_: training.utils.data_utils.collate_fn")
yaml_lines.append("        _partial_: true")
yaml_lines.append("        dict_key: all")
yaml_lines.append("    val:")
yaml_lines.append("      _target_: training.dataset.sam2_datasets.TorchTrainMixedDataset")
yaml_lines.append("      phases_per_epoch: 1")
yaml_lines.append("      batch_sizes:")
yaml_lines.append("        - ${scratch.train_batch_size}")
yaml_lines.append("      datasets:")
yaml_lines.append("        - _target_: training.dataset.utils.RepeatFactorWrapper")
yaml_lines.append("          dataset:")
yaml_lines.append("            _target_: training.dataset.utils.ConcatDataset")
yaml_lines.append("            datasets:")
yaml_lines.append("            - _target_: training.dataset.vos_dataset.VOSDataset")
yaml_lines.append("              transforms: ${vos.val_transforms}")
yaml_lines.append("              training: false")
yaml_lines.append("              video_dataset:")
yaml_lines.append("                _target_: training.dataset.vos_raw_dataset.PNGRawDataset")
yaml_lines.append("                img_folder: ${dataset.img_folder}")
yaml_lines.append("                gt_folder: ${dataset.gt_folder}")
yaml_lines.append("                file_list_txt: ${dataset.val_file_list_txt}")
yaml_lines.append("              sampler:")
yaml_lines.append("                _target_: training.dataset.consecutive_sampler.ConsecutiveFrameSampler")
yaml_lines.append("                num_frames: ${scratch.num_frames}")
yaml_lines.append("                max_num_objects: ${scratch.max_num_objects}")
yaml_lines.append("                max_gap: 2")
yaml_lines.append("                reverse_time_prob: 0.0")
yaml_lines.append("              multiplier: ${dataset.val_multiplier}")
yaml_lines.append("      shuffle: False")
yaml_lines.append("      num_workers: ${scratch.num_train_workers}")
yaml_lines.append("      pin_memory: True")
yaml_lines.append("      drop_last: False")
yaml_lines.append("      collate_fn:")
yaml_lines.append("        _target_: training.utils.data_utils.collate_fn")
yaml_lines.append("        _partial_: true")
yaml_lines.append("        dict_key: syniscopy_val")
yaml_lines.append("")
yaml_lines.append("  optim:")
yaml_lines.append("    amp:")
yaml_lines.append("      enabled: True")
yaml_lines.append("      amp_dtype: bfloat16")
yaml_lines.append("    optimizer:")
yaml_lines.append("      _target_: torch.optim.AdamW")
yaml_lines.append("    gradient_clip:")
yaml_lines.append("      _target_: training.optimizer.GradientClipper")
yaml_lines.append("      max_norm: 0.1")
yaml_lines.append("      norm_type: 2")
yaml_lines.append("    param_group_modifiers:")
yaml_lines.append("      - _target_: training.optimizer.layer_decay_param_modifier")
yaml_lines.append("        _partial_: True")
yaml_lines.append("        layer_decay_value: 0.9")
yaml_lines.append("        apply_to: 'image_encoder.trunk'")
yaml_lines.append("        overrides:")
yaml_lines.append("          - pattern: '*pos_embed*'")
yaml_lines.append("            value: 1.0")
yaml_lines.append("    options:")
yaml_lines.append("      lr:")
yaml_lines.append("        - scheduler:")
yaml_lines.append("            _target_: fvcore.common.param_scheduler.CosineParamScheduler")
yaml_lines.append("            start_value: ${scratch.base_lr}")
yaml_lines.append("            end_value: ${divide:${scratch.base_lr},10}")
yaml_lines.append("        - scheduler:")
yaml_lines.append("            _target_: fvcore.common.param_scheduler.CosineParamScheduler")
yaml_lines.append("            start_value: ${scratch.vision_lr}")
yaml_lines.append("            end_value: ${divide:${scratch.vision_lr},10}")
yaml_lines.append("          param_names:")
yaml_lines.append("            - 'image_encoder.*'")
yaml_lines.append("      weight_decay:")
yaml_lines.append("        - scheduler:")
yaml_lines.append("            _target_: fvcore.common.param_scheduler.ConstantParamScheduler")
yaml_lines.append("            value: 0.1")
yaml_lines.append("        - scheduler:")
yaml_lines.append("            _target_: fvcore.common.param_scheduler.ConstantParamScheduler")
yaml_lines.append("            value: 0.0")
yaml_lines.append("          param_names:")
yaml_lines.append("            - '*bias*'")
yaml_lines.append("          module_cls_names: ['torch.nn.LayerNorm']")
yaml_lines.append("")
yaml_lines.append("  loss:")
yaml_lines.append("    all:")
yaml_lines.append("      _target_: training.loss_fns.MultiStepMultiMasksAndIous")
yaml_lines.append("      weight_dict:")
yaml_lines.append("        loss_mask: 20")
yaml_lines.append("        loss_dice: 1")
yaml_lines.append("        loss_iou: 1")
yaml_lines.append("        loss_class: 1")
yaml_lines.append("      supervise_all_iou: true")
yaml_lines.append("      iou_use_l1_loss: true")
yaml_lines.append("      pred_obj_scores: true")
yaml_lines.append("      focal_gamma_obj_score: 0.0")
yaml_lines.append("      focal_alpha_obj_score: -1.0")
yaml_lines.append("    syniscopy_val:")
yaml_lines.append("      _target_: training.loss_fns.MultiStepMultiMasksAndIous")
yaml_lines.append("      weight_dict:")
yaml_lines.append("        loss_mask: 20")
yaml_lines.append("        loss_dice: 1")
yaml_lines.append("        loss_iou: 1")
yaml_lines.append("        loss_class: 1")
yaml_lines.append("      supervise_all_iou: true")
yaml_lines.append("      iou_use_l1_loss: true")
yaml_lines.append("      pred_obj_scores: true")
yaml_lines.append("      focal_gamma_obj_score: 0.0")
yaml_lines.append("      focal_alpha_obj_score: -1.0")
yaml_lines.append("")
yaml_lines.append("  distributed:")
yaml_lines.append("    backend: nccl")
yaml_lines.append("    find_unused_parameters: True")
yaml_lines.append("")
yaml_lines.append("  logging:")
yaml_lines.append("    tensorboard_writer:")
yaml_lines.append("      _target_: training.utils.logger.make_tensorboard_logger")
yaml_lines.append("      log_dir: ${launcher.experiment_log_dir}/tensorboard")
yaml_lines.append("      flush_secs: 120")
yaml_lines.append("      should_log: True")
yaml_lines.append("    log_dir: ${launcher.experiment_log_dir}/logs")
yaml_lines.append("    log_freq: 10")
yaml_lines.append("")
yaml_lines.append("  checkpoint:")
yaml_lines.append("    save_dir: ${launcher.experiment_log_dir}/checkpoints")
yaml_lines.append("    save_freq: 0")
yaml_lines.append("    model_weight_initializer:")
yaml_lines.append("      _partial_: True")
yaml_lines.append("      _target_: training.utils.checkpoint_utils.load_state_dict_into_model")
yaml_lines.append("      strict: False")
yaml_lines.append("      state_dict:")
yaml_lines.append("        _target_: training.utils.checkpoint_utils.load_checkpoint_and_apply_kernels")
yaml_lines.append(f"        checkpoint_path: {SAM2_DIR / SAM2_BASE_CHECKPOINT_FILENAME}")
yaml_lines.append("        ckpt_state_dict_keys: ['model']")
yaml_lines.append("")
yaml_lines.append("launcher:")
yaml_lines.append("  num_nodes: 1")
yaml_lines.append("  gpus_per_node: 1")
yaml_lines.append(f"  experiment_log_dir: {TRAINING_LOG_DIR}")
yaml_lines.append("")
yaml_lines.append("submitit:")
yaml_lines.append("  use_cluster: false")
yaml_lines.append("  partition: null")
yaml_lines.append("  account: null")
yaml_lines.append("  qos: null")
yaml_lines.append("  cpus_per_task: 10")
yaml_lines.append("  timeout_hour: 24")
yaml_lines.append("  name: null")
yaml_lines.append("  mem_gb: null")
yaml_lines.append("  mem: null")
yaml_lines.append("  constraints: null")
yaml_lines.append("  comment: null")
yaml_lines.append("  srun_args: {}")
yaml_lines.append("  port_range: [10000, 65000]")

with open(CFG_PATH, "w") as f:
    f.write("\n".join(yaml_lines) + "\n")

print("✅ YAML config created")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2/8: Confirm official SAM2 Hydra config location
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("STEP 2/8: Confirming SAM2 Hydra Config Location")
print("=" * 80)

CONFIG_NAME = SAM2_CONFIG_RELATIVE
if not os.path.exists(CFG_PATH):
    raise FileNotFoundError(f"Expected SAM2 training config was not written: {CFG_PATH}")
print(f"✅ Config written under SAM2 config module: {CFG_PATH}")
print(f"✅ Official train.py config name: {CONFIG_NAME}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3/8: Optimize phases_per_epoch
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("STEP 3/8: Optimizing Training Parameters")
print("=" * 80)

DATA_ROOT = str(SAM2_DATASET_ROOT)
LIST_PATH = os.path.join(DATA_ROOT, "train_list.txt")
FRAMES_DIR = os.path.join(DATA_ROOT, "Frames")

with open(LIST_PATH) as f:
    videos = [l.strip() for l in f if l.strip()]

video_frame_counts = []
for vid in videos:
    vid_dir = os.path.join(FRAMES_DIR, vid)
    if os.path.isdir(vid_dir):
        nframes = len(glob.glob(os.path.join(vid_dir, "*.jpg")))
        video_frame_counts.append(nframes)

total_frames = sum(video_frame_counts)
avg_frames = total_frames / len(videos)

multiplier = 20
phases_per_epoch = 1          # Keep SAM2 stock dataset indexing: one mixed-dataset phase per epoch.
num_epochs = 40               # Maximum training budget; checkpoint_best_val.pt selects the reported model.
steps_per_epoch = max(1, len(videos) * multiplier)
total_optimizer_updates = steps_per_epoch * phases_per_epoch * num_epochs
frame_instances_per_epoch = steps_per_epoch * int(SAM2_TRAIN_NUM_FRAMES)
total_frame_instances = frame_instances_per_epoch * num_epochs
frame_equivalent_passes = (total_frame_instances / total_frames) if total_frames else 0.0

print(f"   Videos: {len(videos)}")
print(f"   Total frames: {total_frames}")
print(f"   Avg frames/video: {avg_frames:.1f}")
print(f"   -> phases_per_epoch: {phases_per_epoch}")
print(f"   -> train multiplier: {multiplier}")
print(f"   -> num_epochs: {num_epochs} (maximum budget; best validation checkpoint is used)")
print(f"   -> steps/epoch: {steps_per_epoch}")
print(f"   -> total optimizer updates: {total_optimizer_updates}")
print(f"   -> sampled frame instances: {total_frame_instances}")
print(f"   -> frame-instance equivalent passes: {frame_equivalent_passes:.1f}x")

with open(CFG_PATH, "r") as f:
    cfg = f.read()

cfg = re.sub(r"(^\\s*phases_per_epoch:\s*)\d+", f"\\g<1>{phases_per_epoch}", cfg, flags=re.MULTILINE)
cfg = re.sub(r"(^\\s*multiplier:\s*)\d+", f"\\g<1>{multiplier}", cfg, flags=re.MULTILINE)
cfg = re.sub(r"(^\\s*num_epochs:\s*)\d+", f"\\g<1>{num_epochs}", cfg, flags=re.MULTILINE)

with open(CFG_PATH, "w") as f:
    f.write(cfg)

print("✅ Config optimized")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4/8: Use stock SAM2 dataset indexing
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("STEP 4/8: Using Stock SAM2 Dataset Indexing")
print("=" * 80)

# This training configuration uses SAM2 dataset indexing with one phase per epoch.
if int(phases_per_epoch) != 1:
    raise RuntimeError(
        "This notebook uses SAM2 dataset indexing with phases_per_epoch=1. "
        "Use a Syniscopy-owned dataset wrapper for other epoch semantics."
    )
print("✅ Using stock training/dataset/sam2_datasets.py")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5/8: Install Syniscopy custom supervision/loss support
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("STEP 5/8: Installing Syniscopy Custom Supervision/Loss Support")
print("=" * 80)
print("Keeping SAM2's launcher and stock dataset-mixing path; installing only Syniscopy sidecar/loss behavior.")

RESTORE_SAM2_TRAINING_FILES = [
    "training/train.py",
    "training/utils/data_utils.py",
    "training/dataset/vos_segment_loader.py",
    "training/dataset/vos_dataset.py",
    "training/dataset/transforms.py",
    "training/trainer.py",
    "training/loss_fns.py",
]
subprocess.run(
    ["git", "checkout", "-f", "--", *RESTORE_SAM2_TRAINING_FILES],
    cwd=str(SAM2_DIR),
    check=True,
)
print("✅ Official SAM2 training source restored before Syniscopy support install")

# Inline Syniscopy SAM2 supervision setup.
#
# Syniscopy adds Ignore/ and LossWeight/ sidecars to SAM2 training so
# unsupported pixels contribute zero loss.
from pathlib import Path


SUPERVISION_MARKER = "Syniscopy supervision support"


def _read(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(path)
    return path.read_text()


def _write(path: Path, text: str) -> None:
    path.write_text(text)


def _replace_once(text: str, old: str, new: str, label: str) -> str:
    if old not in text:
        raise RuntimeError(f"Could not update {label}; expected source pattern not found.")
    return text.replace(old, new, 1)


def _replace_top_level_function(text: str, name: str, replacement: str) -> str:
    needle = f"def {name}("
    start = text.find(needle)
    if start < 0:
        raise RuntimeError(f"Could not find function {name} in SAM2 loss_fns.py.")
    end = len(text)
    for marker in ("\ndef ", "\nclass "):
        idx = text.find(marker, start + 1)
        if idx >= 0:
            end = min(end, idx + 1)
    return text[:start] + replacement.rstrip() + "\n\n" + text[end:]


def install_data_utils_support(sam2_root: Path) -> None:
    path = sam2_root / "training" / "utils" / "data_utils.py"
    text = _read(path)
    if f"{SUPERVISION_MARKER}: data_utils" in text:
        return

    text = _replace_once(
        text,
        "    masks: torch.BoolTensor\n    metadata: BatchedVideoMetaData\n\n    dict_key: str\n",
        "    masks: torch.BoolTensor\n"
        "    metadata: BatchedVideoMetaData\n\n"
        "    dict_key: str\n"
        "    ignore_masks: Optional[torch.BoolTensor] = None\n"
        "    loss_weights: Optional[torch.FloatTensor] = None\n",
        "data_utils tensorclass fields",
    )
    text = _replace_once(
        text,
        "    segment: Union[torch.Tensor, dict]  # RLE dict or binary mask\n",
        "    segment: Union[torch.Tensor, dict]  # RLE dict or binary mask\n"
        "    ignore_mask: Optional[torch.Tensor] = None\n"
        "    loss_weight: Optional[torch.Tensor] = None\n",
        "data_utils Object sidecar fields",
    )
    text = _replace_once(
        text,
        "    step_t_masks = [[] for _ in range(T)]\n",
        "    step_t_masks = [[] for _ in range(T)]\n"
        "    step_t_ignore_masks = [[] for _ in range(T)]\n"
        "    step_t_loss_weights = [[] for _ in range(T)]\n",
        "data_utils sidecar lists",
    )
    text = _replace_once(
        text,
        "                step_t_masks[t].append(obj.segment.to(torch.bool))\n",
        "                seg_mask = obj.segment.to(torch.bool)\n"
        "                ignore_mask = getattr(obj, \"ignore_mask\", None)\n"
        "                loss_weight = getattr(obj, \"loss_weight\", None)\n"
        "                if ignore_mask is None:\n"
        "                    ignore_mask = torch.zeros_like(seg_mask, dtype=torch.bool)\n"
        "                else:\n"
        "                    ignore_mask = ignore_mask.to(torch.bool)\n"
        "                if loss_weight is None:\n"
        "                    loss_weight = torch.ones_like(seg_mask, dtype=torch.float32)\n"
        "                else:\n"
        "                    loss_weight = loss_weight.to(torch.float32)\n"
        "                loss_weight = loss_weight.masked_fill(ignore_mask, 0.0)\n"
        "                step_t_masks[t].append(seg_mask)\n"
        "                step_t_ignore_masks[t].append(ignore_mask)\n"
        "                step_t_loss_weights[t].append(loss_weight)\n",
        "data_utils object sidecars",
    )
    text = _replace_once(
        text,
        "    masks = torch.stack([torch.stack(masks, dim=0) for masks in step_t_masks], dim=0)\n",
        "    masks = torch.stack([torch.stack(masks, dim=0) for masks in step_t_masks], dim=0)\n"
        "    ignore_masks = torch.stack(\n"
        "        [torch.stack(masks, dim=0) for masks in step_t_ignore_masks], dim=0\n"
        "    )\n"
        "    loss_weights = torch.stack(\n"
        "        [torch.stack(weights, dim=0) for weights in step_t_loss_weights], dim=0\n"
        "    )\n",
        "data_utils sidecar stacks",
    )
    text = _replace_once(
        text,
        "        dict_key=dict_key,\n",
        "        dict_key=dict_key,\n"
        "        ignore_masks=ignore_masks,\n"
        "        loss_weights=loss_weights,\n",
        "data_utils return fields",
    )
    text = _replace_once(
        text,
        "        metadata=BatchedVideoMetaData(\n"
        "            unique_objects_identifier=objects_identifier,\n"
        "            frame_orig_size=frame_orig_size,\n"
        "        ),\n",
        "        metadata=BatchedVideoMetaData(\n"
        "            unique_objects_identifier=objects_identifier,\n"
        "            frame_orig_size=frame_orig_size,\n"
        "            batch_size=[T, objects_identifier.shape[1]],\n"
        "        ),\n",
        "data_utils metadata batch size",
    )
    text = f"# {SUPERVISION_MARKER}: data_utils\n" + text
    _write(path, text)


def install_segment_loader_support(sam2_root: Path) -> None:
    path = sam2_root / "training" / "dataset" / "vos_segment_loader.py"
    text = _read(path)
    if f"{SUPERVISION_MARKER}: vos_segment_loader" in text:
        return

    text = _replace_once(
        text,
        "        self.video_png_root = video_png_root\n",
        "        self.video_png_root = video_png_root\n"
        "        _dataset_root = os.path.dirname(os.path.dirname(video_png_root))\n"
        "        _video_name = os.path.basename(video_png_root)\n"
        "        self.ignore_png_root = os.path.join(_dataset_root, \"Ignore\", _video_name)\n"
        "        self.loss_weight_png_root = os.path.join(_dataset_root, \"LossWeight\", _video_name)\n",
        "PalettisedPNGSegmentLoader sidecar roots",
    )
    helper = '''
    def _load_syniscopy_sidecars(self, frame_id, mask_shape):
        filename = self.frame_id_to_png_filename[frame_id]
        ignore = None
        weight = None
        ignore_path = os.path.join(self.ignore_png_root, filename)
        if os.path.exists(ignore_path):
            ignore_arr = np.array(PILImage.open(ignore_path).convert("L"))
            ignore = torch.from_numpy(ignore_arr > 0)
        weight_path = os.path.join(self.loss_weight_png_root, filename)
        if os.path.exists(weight_path):
            weight_arr = np.array(PILImage.open(weight_path).convert("L"))
            weight = torch.from_numpy(weight_arr.astype(np.float32) / 255.0)
        if ignore is not None and tuple(ignore.shape) != tuple(mask_shape):
            raise ValueError("Syniscopy ignore sidecar shape does not match GT mask.")
        if weight is not None and tuple(weight.shape) != tuple(mask_shape):
            raise ValueError("Syniscopy loss-weight sidecar shape does not match GT mask.")
        return ignore, weight

'''
    text = _replace_once(
        text,
        "    def __len__(self):\n        return\n",
        helper + "    def __len__(self):\n        return\n",
        "PalettisedPNGSegmentLoader helper",
    )
    text = _replace_once(
        text,
        "            binary_segments[i] = torch.from_numpy(bs)\n",
        "            segment = torch.from_numpy(bs)\n"
        "            ignore, weight = self._load_syniscopy_sidecars(frame_id, segment.shape)\n"
        "            if ignore is not None or weight is not None:\n"
        "                binary_segments[i] = {\n"
        "                    \"mask\": segment,\n"
        "                    \"ignore_mask\": ignore,\n"
        "                    \"loss_weight\": weight,\n"
        "                }\n"
        "            else:\n"
        "                binary_segments[i] = segment\n",
        "PalettisedPNGSegmentLoader segment dict",
    )
    text = f"# {SUPERVISION_MARKER}: vos_segment_loader\n" + text
    _write(path, text)


def install_transforms_support(sam2_root: Path) -> None:
    path = sam2_root / "training" / "dataset" / "transforms.py"
    text = _read(path)
    if f"{SUPERVISION_MARKER}: transforms" in text:
        return

    helper = '''
def _apply_syniscopy_sidecar_transform(obj, transform):
    if hasattr(obj, "ignore_mask") and obj.ignore_mask is not None:
        obj.ignore_mask = transform(obj.ignore_mask)
    if hasattr(obj, "loss_weight") and obj.loss_weight is not None:
        obj.loss_weight = transform(obj.loss_weight)


'''
    text = _replace_once(
        text,
        "\ndef hflip(datapoint, index):\n",
        helper + "def hflip(datapoint, index):\n",
        "transforms sidecar helper",
    )
    text = _replace_once(
        text,
        "        if obj.segment is not None:\n            obj.segment = F.hflip(obj.segment)\n",
        "        if obj.segment is not None:\n"
        "            obj.segment = F.hflip(obj.segment)\n"
        "            _apply_syniscopy_sidecar_transform(obj, F.hflip)\n",
        "transforms hflip sidecars",
    )
    text = _replace_once(
        text,
        "        if obj.segment is not None:\n            obj.segment = F.resize(obj.segment[None, None], size).squeeze()\n",
        "        if obj.segment is not None:\n"
        "            obj.segment = F.resize(obj.segment[None, None], size).squeeze()\n"
        "            _apply_syniscopy_sidecar_transform(\n"
        "                obj,\n"
        "                lambda sidecar: F.resize(\n"
        "                    sidecar[None, None],\n"
        "                    size,\n"
        "                    interpolation=InterpolationMode.NEAREST,\n"
        "                ).squeeze(),\n"
        "            )\n",
        "transforms resize sidecars",
    )
    text = _replace_once(
        text,
        "            this_masks = [\n"
        "                obj.segment.unsqueeze(0) if obj.segment is not None else None\n"
        "                for obj in img.objects\n"
        "            ]\n",
        "            this_masks = [\n"
        "                obj.segment.unsqueeze(0) if obj.segment is not None else None\n"
        "                for obj in img.objects\n"
        "            ]\n"
        "            this_ignore_masks = [\n"
        "                obj.ignore_mask.unsqueeze(0)\n"
        "                if getattr(obj, \"ignore_mask\", None) is not None\n"
        "                else None\n"
        "                for obj in img.objects\n"
        "            ]\n"
        "            this_loss_weights = [\n"
        "                obj.loss_weight.unsqueeze(0)\n"
        "                if getattr(obj, \"loss_weight\", None) is not None\n"
        "                else None\n"
        "                for obj in img.objects\n"
        "            ]\n",
        "transforms affine sidecar inputs",
    )
    text = _replace_once(
        text,
        "            transformed_bboxes, transformed_masks = [], []\n",
        "            transformed_bboxes, transformed_masks = [], []\n"
        "            transformed_ignore_masks, transformed_loss_weights = [], []\n",
        "transforms affine sidecar outputs",
    )
    text = _replace_once(
        text,
        "                if this_masks[i] is None:\n                    transformed_masks.append(None)\n                    # Dummy bbox for a dummy target\n                    transformed_bboxes.append(torch.tensor([[0, 0, 1, 1]]))\n",
        "                if this_masks[i] is None:\n"
        "                    transformed_masks.append(None)\n"
        "                    transformed_ignore_masks.append(None)\n"
        "                    transformed_loss_weights.append(None)\n"
        "                    # Dummy bbox for a dummy target\n"
        "                    transformed_bboxes.append(torch.tensor([[0, 0, 1, 1]]))\n",
        "transforms affine empty sidecars",
    )
    text = _replace_once(
        text,
        "                    transformed_masks.append(transformed_mask.squeeze())\n\n            for i in range(len(img.objects)):\n                img.objects[i].segment = transformed_masks[i]\n",
        "                    transformed_masks.append(transformed_mask.squeeze())\n"
        "                    if this_ignore_masks[i] is None:\n"
        "                        transformed_ignore_masks.append(None)\n"
        "                    else:\n"
        "                        transformed_ignore_masks.append(\n"
        "                            F.affine(\n"
        "                                this_ignore_masks[i],\n"
        "                                *affine_params,\n"
        "                                interpolation=InterpolationMode.NEAREST,\n"
        "                                fill=1.0,\n"
        "                            ).squeeze()\n"
        "                        )\n"
        "                    if this_loss_weights[i] is None:\n"
        "                        transformed_loss_weights.append(None)\n"
        "                    else:\n"
        "                        transformed_loss_weights.append(\n"
        "                            F.affine(\n"
        "                                this_loss_weights[i],\n"
        "                                *affine_params,\n"
        "                                interpolation=InterpolationMode.NEAREST,\n"
        "                                fill=0.0,\n"
        "                            ).squeeze()\n"
        "                        )\n\n"
        "            for i in range(len(img.objects)):\n"
        "                img.objects[i].segment = transformed_masks[i]\n"
        "                if hasattr(img.objects[i], \"ignore_mask\"):\n"
        "                    img.objects[i].ignore_mask = transformed_ignore_masks[i]\n"
        "                if hasattr(img.objects[i], \"loss_weight\"):\n"
        "                    img.objects[i].loss_weight = transformed_loss_weights[i]\n",
        "transforms affine sidecar assignment",
    )
    text = f"# {SUPERVISION_MARKER}: transforms\n" + text
    _write(path, text)


def install_vos_dataset_support(sam2_root: Path) -> None:
    path = sam2_root / "training" / "dataset" / "vos_dataset.py"
    text = _read(path)
    if f"{SUPERVISION_MARKER}: vos_dataset" in text:
        return

    text = _replace_once(
        text,
        "                    # segment is uint8 and remains uint8 throughout the transforms\n                    segment = segments[obj_id].to(torch.uint8)\n",
        "                    # segment is uint8 and remains uint8 throughout the transforms\n"
        "                    raw_segment = segments[obj_id]\n"
        "                    ignore_mask = None\n"
        "                    loss_weight = None\n"
        "                    if isinstance(raw_segment, dict):\n"
        "                        segment = raw_segment[\"mask\"].to(torch.uint8)\n"
        "                        if raw_segment.get(\"ignore_mask\") is not None:\n"
        "                            ignore_mask = raw_segment.get(\"ignore_mask\").to(torch.uint8)\n"
        "                        if raw_segment.get(\"loss_weight\") is not None:\n"
        "                            loss_weight = raw_segment.get(\"loss_weight\").to(torch.float32)\n"
        "                    else:\n"
        "                        segment = raw_segment.to(torch.uint8)\n",
        "VOSDataset sidecar extraction",
    )
    text = _replace_once(
        text,
        "                    segment = torch.zeros(h, w, dtype=torch.uint8)\n",
        "                    segment = torch.zeros(h, w, dtype=torch.uint8)\n"
        "                    ignore_mask = None\n"
        "                    loss_weight = None\n",
        "VOSDataset missing-target sidecars",
    )
    text = _replace_once(
        text,
        "                        segment=segment,\n",
        "                        segment=segment,\n"
        "                        ignore_mask=ignore_mask,\n"
        "                        loss_weight=loss_weight,\n",
        "VOSDataset Object sidecars",
    )
    text = f"# {SUPERVISION_MARKER}: vos_dataset\n" + text
    _write(path, text)


def install_trainer_support(sam2_root: Path) -> None:
    path = sam2_root / "training" / "trainer.py"
    text = _read(path)
    if f"{SUPERVISION_MARKER}: trainer" in text:
        return

    text = _replace_once(
        text,
        "        targets = batch.masks\n",
        "        if getattr(batch, \"ignore_masks\", None) is not None or getattr(batch, \"loss_weights\", None) is not None:\n"
        "            targets = {\n"
        "                \"masks\": batch.masks,\n"
        "                \"ignore_masks\": getattr(batch, \"ignore_masks\", None),\n"
        "                \"loss_weights\": getattr(batch, \"loss_weights\", None),\n"
        "            }\n"
        "        else:\n"
        "            targets = batch.masks\n",
        "trainer target sidecars",
    )
    best_val_helper = r'''
    def _maybe_save_syniscopy_best_val_checkpoint(self, val_stats):
        if self.distributed_rank != 0:
            return
        val_loss_keys = [
            key for key in val_stats
            if key.startswith("Losses/val_") and key.endswith("_loss")
        ]
        if not val_loss_keys:
            return
        preferred_key = "Losses/val_syniscopy_val_loss"
        loss_key = preferred_key if preferred_key in val_stats else sorted(val_loss_keys)[0]
        val_loss = float(val_stats[loss_key])
        current_best = self.best_meter_values.get("syniscopy_best_val_loss")
        if current_best is not None and val_loss >= float(current_best) - self.EPSILON:
            return
        checkpoint_path = os.path.join(self.checkpoint_conf.save_dir, "checkpoint.pt")
        if not g_pathmgr.exists(checkpoint_path):
            logging.warning("Validation improved, but checkpoint.pt is not available to copy.")
            return
        best_path = os.path.join(self.checkpoint_conf.save_dir, "checkpoint_best_val.pt")
        tmp_path = f"{best_path}.tmp"
        if g_pathmgr.exists(tmp_path):
            g_pathmgr.rm(tmp_path)
        if g_pathmgr.exists(best_path):
            g_pathmgr.rm(best_path)
        g_pathmgr.copy(checkpoint_path, tmp_path)
        assert g_pathmgr.mv(tmp_path, best_path)
        self.best_meter_values["syniscopy_best_val_loss"] = val_loss
        self.best_meter_values["syniscopy_best_val_loss_key"] = loss_key
        metadata = {
            "selection_metric": loss_key,
            "selection_mode": "min",
            "best_val_loss": val_loss,
            "trainer_epoch": int(self.epoch),
            "checkpoint_epoch": int(self.epoch) + 1,
            "source_checkpoint": checkpoint_path,
            "best_checkpoint": best_path,
        }
        metadata_path = os.path.join(self.checkpoint_conf.save_dir, "checkpoint_best_val.json")
        with g_pathmgr.open(metadata_path, "w") as f:
            f.write(json.dumps(metadata, indent=2) + "\n")
        logging.info(f"Saved best validation checkpoint: {metadata}")

'''
    text = _replace_once(
        text,
        "    def val_epoch(self, val_loader, phase):\n",
        best_val_helper + "    def val_epoch(self, val_loader, phase):\n",
        "trainer best-val helper",
    )
    text = _replace_once(
        text,
        "                os.path.join(self.logging_conf.log_dir, \"val_stats.json\"),\n"
        "                \"a\",\n"
        "            ) as f:\n"
        "                f.write(json.dumps(outs) + \"\\n\")\n",
        "                os.path.join(self.logging_conf.log_dir, \"val_stats.json\"),\n"
        "                \"a\",\n"
        "            ) as f:\n"
        "                f.write(json.dumps(outs) + \"\\n\")\n"
        "            self._maybe_save_syniscopy_best_val_checkpoint(outs)\n",
        "trainer best-val call",
    )
    text = text.replace(
        "torch.cuda.amp.autocast(",
        "torch.amp.autocast(\"cuda\", ",
    )
    text = f"# {SUPERVISION_MARKER}: trainer\n" + text
    _write(path, text)


WEIGHTED_DICE = '''def dice_loss(inputs, targets, num_objects, loss_on_multimask=False, pixel_weight=None):
    """Weighted DICE loss; Syniscopy ignore pixels carry zero weight."""
    inputs = inputs.sigmoid()
    if loss_on_multimask:
        assert inputs.dim() == 4 and targets.dim() == 4
        inputs = inputs.flatten(2)
        targets = targets.flatten(2)
        if pixel_weight is None:
            weight = torch.ones_like(targets)
        else:
            weight = pixel_weight.flatten(2).to(inputs.dtype)
        numerator = 2 * (weight * inputs * targets).sum(-1)
        denominator = (weight * inputs).sum(-1) + (weight * targets).sum(-1)
        loss = 1 - (numerator + 1) / (denominator + 1)
        return loss / num_objects
    inputs = inputs.flatten(1)
    targets = targets.flatten(1)
    if pixel_weight is None:
        weight = torch.ones_like(targets)
    else:
        weight = pixel_weight.flatten(1).to(inputs.dtype)
    numerator = 2 * (weight * inputs * targets).sum(1)
    denominator = (weight * inputs).sum(1) + (weight * targets).sum(1)
    loss = 1 - (numerator + 1) / (denominator + 1)
    return loss.sum() / num_objects
'''


WEIGHTED_FOCAL = '''def sigmoid_focal_loss(
    inputs,
    targets,
    num_objects,
    alpha: float = 0.25,
    gamma: float = 2,
    loss_on_multimask=False,
    pixel_weight=None,
):
    """Weighted sigmoid focal loss; zero-weight pixels contribute no loss."""
    prob = inputs.sigmoid()
    ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction="none")
    p_t = prob * targets + (1 - prob) * (1 - targets)
    loss = ce_loss * ((1 - p_t) ** gamma)

    if alpha >= 0:
        alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
        loss = alpha_t * loss

    if pixel_weight is None:
        pixel_weight = torch.ones_like(loss)
    else:
        pixel_weight = pixel_weight.to(loss.dtype).expand_as(loss)
    loss = loss * pixel_weight

    if loss_on_multimask:
        assert loss.dim() == 4
        denom = pixel_weight.flatten(2).sum(-1).clamp(min=1.0)
        return loss.flatten(2).sum(-1) / denom / num_objects
    denom = pixel_weight.flatten(1).sum(1).clamp(min=1.0)
    return (loss.flatten(1).sum(1) / denom).sum() / num_objects
'''


WEIGHTED_IOU = '''def iou_loss(
    inputs,
    targets,
    pred_ious,
    num_objects,
    loss_on_multimask=False,
    use_l1_loss=False,
    pixel_weight=None,
):
    """IoU loss computed only over valid Syniscopy-supervised pixels."""
    assert inputs.dim() == 4 and targets.dim() == 4
    pred_mask = inputs.flatten(2) > 0
    gt_mask = targets.flatten(2) > 0
    if pixel_weight is None:
        valid = torch.ones_like(gt_mask, dtype=torch.bool)
    else:
        valid = pixel_weight.flatten(2).expand_as(gt_mask) > 0
    area_i = torch.sum((pred_mask & gt_mask) & valid, dim=-1).float()
    area_u = torch.sum((pred_mask | gt_mask) & valid, dim=-1).float()
    actual_ious = area_i / torch.clamp(area_u, min=1.0)
    if use_l1_loss:
        loss = F.l1_loss(pred_ious, actual_ious, reduction="none")
    else:
        loss = F.mse_loss(pred_ious, actual_ious, reduction="none")
    if loss_on_multimask:
        return loss / num_objects
    return loss.sum() / num_objects
'''


def install_loss_fns_support(sam2_root: Path) -> None:
    path = sam2_root / "training" / "loss_fns.py"
    text = _read(path)
    if f"{SUPERVISION_MARKER}: loss_fns" in text:
        return

    text = _replace_top_level_function(text, "dice_loss", WEIGHTED_DICE)
    text = _replace_top_level_function(text, "sigmoid_focal_loss", WEIGHTED_FOCAL)
    text = _replace_top_level_function(text, "iou_loss", WEIGHTED_IOU)

    text = _replace_once(
        text,
        "    def forward(self, outs_batch: List[Dict], targets_batch: torch.Tensor):\n"
        "        assert len(outs_batch) == len(targets_batch)\n"
        "        num_objects = torch.tensor(\n"
        "            (targets_batch.shape[1]), device=targets_batch.device, dtype=torch.float\n"
        "        )  # Number of objects is fixed within a batch\n",
        "    def forward(self, outs_batch: List[Dict], targets_batch: torch.Tensor):\n"
        "        if isinstance(targets_batch, dict):\n"
        "            masks_batch = targets_batch[\"masks\"]\n"
        "            ignore_masks_batch = targets_batch.get(\"ignore_masks\")\n"
        "            loss_weights_batch = targets_batch.get(\"loss_weights\")\n"
        "        else:\n"
        "            masks_batch = targets_batch\n"
        "            ignore_masks_batch = None\n"
        "            loss_weights_batch = None\n"
        "        assert len(outs_batch) == len(masks_batch)\n"
        "        num_objects = torch.tensor(\n"
        "            (masks_batch.shape[1]), device=masks_batch.device, dtype=torch.float\n"
        "        )  # Number of objects is fixed within a batch\n",
        "loss_fns forward header",
    )
    text = _replace_once(
        text,
        "        for outs, targets in zip(outs_batch, targets_batch):\n"
        "            cur_losses = self._forward(outs, targets, num_objects)\n",
        "        ignore_iter = ignore_masks_batch if ignore_masks_batch is not None else [None] * len(masks_batch)\n"
        "        weight_iter = loss_weights_batch if loss_weights_batch is not None else [None] * len(masks_batch)\n"
        "        for outs, targets, ignore_masks, loss_weights in zip(\n"
        "            outs_batch, masks_batch, ignore_iter, weight_iter\n"
        "        ):\n"
        "            cur_losses = self._forward(\n"
        "                outs, targets, num_objects, ignore_masks=ignore_masks, loss_weights=loss_weights\n"
        "            )\n",
        "loss_fns forward loop",
    )
    text = _replace_once(
        text,
        "    def _forward(self, outputs: Dict, targets: torch.Tensor, num_objects):\n",
        "    def _forward(self, outputs: Dict, targets: torch.Tensor, num_objects, ignore_masks=None, loss_weights=None):\n",
        "loss_fns _forward signature",
    )
    text = _replace_once(
        text,
        "        target_masks = targets.unsqueeze(1).float()\n        assert target_masks.dim() == 4  # [N, 1, H, W]\n",
        "        target_masks = targets.unsqueeze(1).float()\n"
        "        assert target_masks.dim() == 4  # [N, 1, H, W]\n"
        "        ignore_masks = ignore_masks.unsqueeze(1).bool() if ignore_masks is not None else None\n"
        "        loss_weights = loss_weights.unsqueeze(1).float() if loss_weights is not None else None\n",
        "loss_fns sidecar unsqueeze",
    )
    text = _replace_once(
        text,
        "            self._update_losses(\n"
        "                losses, src_masks, target_masks, ious, num_objects, object_score_logits\n"
        "            )\n",
        "            self._update_losses(\n"
        "                losses,\n"
        "                src_masks,\n"
        "                target_masks,\n"
        "                ious,\n"
        "                num_objects,\n"
        "                object_score_logits,\n"
        "                ignore_masks=ignore_masks,\n"
        "                loss_weights=loss_weights,\n"
        "            )\n",
        "loss_fns update call",
    )
    text = _replace_once(
        text,
        "    def _update_losses(\n"
        "        self, losses, src_masks, target_masks, ious, num_objects, object_score_logits\n"
        "    ):\n"
        "        target_masks = target_masks.expand_as(src_masks)\n",
        "    def _update_losses(\n"
        "        self,\n"
        "        losses,\n"
        "        src_masks,\n"
        "        target_masks,\n"
        "        ious,\n"
        "        num_objects,\n"
        "        object_score_logits,\n"
        "        ignore_masks=None,\n"
        "        loss_weights=None,\n"
        "    ):\n"
        "        target_masks = target_masks.expand_as(src_masks)\n"
        "        if loss_weights is None:\n"
        "            pixel_weight = torch.ones_like(target_masks, dtype=src_masks.dtype)\n"
        "        else:\n"
        "            positive_weight = loss_weights.to(src_masks.dtype).expand_as(src_masks)\n"
        "            pixel_weight = torch.ones_like(target_masks, dtype=src_masks.dtype)\n"
        "            pixel_weight = torch.where(target_masks > 0, positive_weight, pixel_weight)\n"
        "        if ignore_masks is not None:\n"
        "            pixel_weight = pixel_weight.masked_fill(ignore_masks.expand_as(src_masks), 0.0)\n",
        "loss_fns update signature",
    )
    text = _replace_once(
        text,
        "            loss_on_multimask=True,\n        )\n        loss_multidice = dice_loss(\n            src_masks, target_masks, num_objects, loss_on_multimask=True\n        )\n",
        "            loss_on_multimask=True,\n"
        "            pixel_weight=pixel_weight,\n"
        "        )\n"
        "        loss_multidice = dice_loss(\n"
        "            src_masks,\n"
        "            target_masks,\n"
        "            num_objects,\n"
        "            loss_on_multimask=True,\n"
        "            pixel_weight=pixel_weight,\n"
        "        )\n",
        "loss_fns focal/dice pixel weights",
    )
    text = _replace_once(
        text,
        "            use_l1_loss=self.iou_use_l1_loss,\n        )\n",
        "            use_l1_loss=self.iou_use_l1_loss,\n"
        "            pixel_weight=pixel_weight,\n"
        "        )\n",
        "loss_fns iou pixel weights",
    )
    text = f"# {SUPERVISION_MARKER}: loss_fns\n" + text
    _write(path, text)


def install_syniscopy_supervision_support(sam2_root: str | Path) -> None:
    root = Path(sam2_root)
    install_data_utils_support(root)
    install_segment_loader_support(root)
    install_transforms_support(root)
    install_vos_dataset_support(root)
    install_trainer_support(root)
    install_loss_fns_support(root)

install_syniscopy_supervision_support('/content/segment-anything-2')
_loss_code = (SAM2_DIR / 'training' / 'loss_fns.py').read_text()
if 'torch.where(target_masks > 0, positive_weight, pixel_weight)' not in _loss_code:
    raise RuntimeError(
        'Syniscopy loss patch is unsafe: background pixels must stay valid while '
        'LossWeight only modulates positive target pixels.'
    )
print('✅ Syniscopy Ignore/LossWeight SAM2 training support installed')


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6/8: Pre-training validation
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("STEP 6/8: Pre-Training Validation")
print("=" * 80)

dataset_path = str(SAM2_DATASET_ROOT)
train_list_txt = os.path.join(dataset_path, "train_list.txt")
val_list_txt = os.path.join(dataset_path, "val_list.txt")

with open(train_list_txt, 'r') as f:
    train_videos = [line.strip() for line in f if line.strip()]
with open(val_list_txt, 'r') as f:
    val_videos = [line.strip() for line in f if line.strip()]

def _count_frames(video_ids):
    total = 0
    for video in video_ids:
        frame_dir = os.path.join(dataset_path, "Frames", video)
        if os.path.exists(frame_dir):
            total += len(glob.glob(os.path.join(frame_dir, "*.jpg")))
    return total

train_frames = _count_frames(train_videos)
val_frames = _count_frames(val_videos)

print(f"   Training videos: {len(train_videos)}")
print(f"   Validation videos: {len(val_videos)}")
print(f"   Training frames: {train_frames}")
print(f"   Validation frames: {val_frames}")

if len(train_videos) < 1 or len(val_videos) < 1:
    raise ValueError("Need at least one training video and one validation video")

print("   ✅ Dataset split OK")

with open(CFG_PATH, 'r') as f:
    config = yaml.safe_load(f)

phases = config['scratch']['phases_per_epoch']
multiplier_val = config['dataset']['multiplier']
val_multiplier_val = config['dataset']['val_multiplier']
num_epochs_val = config['scratch']['num_epochs']
mode_val = config['trainer']['mode']

print(f"   phases_per_epoch: {phases}")
print(f"   train multiplier: {multiplier_val}")
print(f"   validation multiplier: {val_multiplier_val}")
print(f"   num_epochs: {num_epochs_val}")
print(f"   trainer mode: {mode_val}")
if mode_val != "train":
    raise ValueError("SAM2 trainer mode must be train so validation runs each epoch")
train_dict_key = config['trainer']['data']['train']['collate_fn']['dict_key']
val_dict_key = config['trainer']['data']['val']['collate_fn']['dict_key']
all_loss_keys = set(config['trainer']['loss'].keys())
val_loss_keys = all_loss_keys - {'all'}
print(f"   train dict_key: {train_dict_key}")
print(f"   validation dict_key: {val_dict_key}")
print(f"   loss keys: {sorted(all_loss_keys)}")
if train_dict_key not in all_loss_keys:
    raise ValueError(
        "SAM2 Trainer requires train collate_fn.dict_key to match a loss key. "
        f"Got train dict_key={train_dict_key!r}, loss keys={sorted(all_loss_keys)}"
    )
if val_dict_key == 'all' or val_dict_key not in val_loss_keys:
    raise ValueError(
        "SAM2 Trainer requires validation collate_fn.dict_key to match a non-'all' loss key. "
        f"Got val dict_key={val_dict_key!r}, non-all loss keys={sorted(val_loss_keys)}"
    )
print("   ✅ Config OK")

ckpt_path = str(SAM2_DIR / SAM2_BASE_CHECKPOINT_FILENAME)
if not os.path.exists(ckpt_path):
    raise FileNotFoundError("Checkpoint missing")

size_mb = os.path.getsize(ckpt_path) / (1024 * 1024)
print(f"   Checkpoint size: {size_mb:.1f} MB")
print("   ✅ Checkpoint OK")

import importlib, sys
sam2_root_str = str(SAM2_DIR)
if sam2_root_str not in sys.path:
    sys.path.insert(0, sam2_root_str)
syniscopy_support_modules = [
    "training.utils.data_utils",
    "training.dataset.vos_segment_loader",
    "training.dataset.transforms",
    "training.dataset.vos_dataset",
    "training.trainer",
    "training.loss_fns",
]
importlib.invalidate_caches()
for module_name in syniscopy_support_modules:
    sys.modules.pop(module_name, None)
for module_name in syniscopy_support_modules:
    importlib.import_module(module_name)
print("   ✅ Syniscopy SAM2 support module imports OK")

print("   Checking one transformed SAM2 training sample before model launch...")
from hydra.utils import instantiate
from omegaconf import OmegaConf
from training.utils.data_utils import collate_fn

_preflight_cfg = OmegaConf.load(CFG_PATH)
_preflight_dataset_cfg = _preflight_cfg.trainer.data.train.datasets[0].dataset.datasets[0]
_preflight_dataset = instantiate(_preflight_dataset_cfg, _recursive_=True)
_preflight_sample = _preflight_dataset[0]
_preflight_train_dict_key = str(_preflight_cfg.trainer.data.train.collate_fn.dict_key)
_preflight_batch = collate_fn([_preflight_sample], dict_key=_preflight_train_dict_key)
if _preflight_batch.dict_key != _preflight_train_dict_key:
    raise RuntimeError(
        f"train preflight dict_key mismatch: {_preflight_batch.dict_key} != {_preflight_train_dict_key}"
    )
if getattr(_preflight_batch, "ignore_masks", None) is None:
    raise RuntimeError("Syniscopy ignore_masks missing from preflight batch")
if getattr(_preflight_batch, "loss_weights", None) is None:
    raise RuntimeError("Syniscopy loss_weights missing from preflight batch")
if _preflight_batch.ignore_masks.shape != _preflight_batch.masks.shape:
    raise RuntimeError(
        f"ignore_masks shape {_preflight_batch.ignore_masks.shape} != masks shape {_preflight_batch.masks.shape}"
    )
if _preflight_batch.loss_weights.shape != _preflight_batch.masks.shape:
    raise RuntimeError(
        f"loss_weights shape {_preflight_batch.loss_weights.shape} != masks shape {_preflight_batch.masks.shape}"
    )
_preflight_val_dataset_cfg = _preflight_cfg.trainer.data.val.datasets[0].dataset.datasets[0]
_preflight_val_dataset = instantiate(_preflight_val_dataset_cfg, _recursive_=True)
_preflight_val_sample = _preflight_val_dataset[0]
_preflight_val_dict_key = str(_preflight_cfg.trainer.data.val.collate_fn.dict_key)
_preflight_val_batch = collate_fn([_preflight_val_sample], dict_key=_preflight_val_dict_key)
if _preflight_val_batch.dict_key != _preflight_val_dict_key:
    raise RuntimeError(
        f"validation preflight dict_key mismatch: {_preflight_val_batch.dict_key} != {_preflight_val_dict_key}"
    )
if getattr(_preflight_val_batch, "ignore_masks", None) is None:
    raise RuntimeError("Syniscopy validation ignore_masks missing from preflight batch")
if getattr(_preflight_val_batch, "loss_weights", None) is None:
    raise RuntimeError("Syniscopy validation loss_weights missing from preflight batch")
if _preflight_val_batch.ignore_masks.shape != _preflight_val_batch.masks.shape:
    raise RuntimeError(
        f"validation ignore_masks shape {_preflight_val_batch.ignore_masks.shape} != masks shape {_preflight_val_batch.masks.shape}"
    )
if _preflight_val_batch.loss_weights.shape != _preflight_val_batch.masks.shape:
    raise RuntimeError(
        f"validation loss_weights shape {_preflight_val_batch.loss_weights.shape} != masks shape {_preflight_val_batch.masks.shape}"
    )
del _preflight_dataset, _preflight_sample, _preflight_batch
del _preflight_val_dataset, _preflight_val_sample, _preflight_val_batch
print("   ✅ SAM2 train/validation transform-collate preflight batches OK")

print("\n✅ ALL PRE-TRAINING CHECKS PASSED")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 7/8: Checkpoint Management (single directory, no duplication)
# ═══════════════════════════════════════════════════════════════════════════════
# Preserve Drive-backed checkpoints by default. Official SAM2 resumes from
# checkpoint.pt inside checkpoint.save_dir; deleting .pt files here would defeat
# disconnect recovery. Set RESET_TRAINING_CHECKPOINTS=True only for an intentional
# clean restart with the same MICROSCOPE_LABEL.
old_ckpt_dir = str(TRAINING_CHECKPOINT_DIR)
if RESET_TRAINING_CHECKPOINTS and os.path.exists(old_ckpt_dir):
    existing = [f for f in os.listdir(old_ckpt_dir) if f.endswith(".pt")]
    if existing:
        print(f"⚠️  RESET_TRAINING_CHECKPOINTS=True; removing {len(existing)} checkpoint file(s)")
        for f in existing:
            os.remove(os.path.join(old_ckpt_dir, f))
        print("✅ Prior checkpoints cleared for a fresh restart")
elif os.path.exists(os.path.join(old_ckpt_dir, "checkpoint.pt")):
    print("✅ Existing checkpoint.pt preserved; SAM2 can resume from it")
elif os.path.exists(old_ckpt_dir):
    existing = [f for f in os.listdir(old_ckpt_dir) if f.endswith(".pt")]
    if existing:
        print("ℹ️ Existing numbered checkpoints preserved; checkpoint.pt will be restored below if needed")
else:
    print("ℹ️ No checkpoint directory yet - starting fresh unless SAM2 creates one")


print("\n" + "=" * 80)
print("STEP 7/8: Setting Up Checkpoint Management")
print("=" * 80)

CHECKPOINT_DIR = str(TRAINING_CHECKPOINT_DIR)
OLD_BACKUP_DIR = str(CONFIG_DIR / "canonical_checkpoints")

CHECKPOINT_PATTERN = re.compile(r"checkpoint_(\d+)\.pt$")
MAX_CHECKPOINTS_TO_KEEP = 0

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

LOSS_SEMANTICS_MARKER_PATH = os.path.join(CHECKPOINT_DIR, "syniscopy_loss_semantics_version.txt")

def _read_text_if_exists(path):
    try:
        if path and os.path.exists(path):
            with open(path, "r") as fh:
                return fh.read().strip()
    except Exception:
        return None
    return None

def _manifest_loss_semantics(path):
    try:
        if path and os.path.exists(path):
            with open(path, "r") as fh:
                return json.load(fh).get("sam2_loss_semantics_version")
    except Exception:
        return None
    return None

_existing_training_pts = []
if os.path.exists(CHECKPOINT_DIR):
    _existing_training_pts = [f for f in os.listdir(CHECKPOINT_DIR) if f.endswith(".pt")]
_existing_final_checkpoint = os.path.exists(str(FINAL_CHECKPOINT_PATH))
if not RESET_TRAINING_CHECKPOINTS and (_existing_training_pts or _existing_final_checkpoint):
    _known_versions = {
        _read_text_if_exists(LOSS_SEMANTICS_MARKER_PATH),
        _manifest_loss_semantics(str(CHECKPOINT_MANIFEST_PATH)),
    }
    _known_versions.discard(None)
    if SAM2_LOSS_SEMANTICS_VERSION not in _known_versions:
        raise RuntimeError(
            "Existing SAM2 checkpoints were found, but they do not declare the current "
            f"loss semantics {SAM2_LOSS_SEMANTICS_VERSION!r}. Set "
            "RESET_TRAINING_CHECKPOINTS=True for a fresh E08 training run, or use a new "
            "MICROSCOPE_LABEL. Do not resume an old checkpoint across this loss fix."
        )
with open(LOSS_SEMANTICS_MARKER_PATH, "w") as _fh:
    _fh.write(SAM2_LOSS_SEMANTICS_VERSION + "\n")
print("   Loss semantics:", SAM2_LOSS_SEMANTICS_VERSION)

from google.colab import auth
from googleapiclient.discovery import build

EMPTY_GOOGLE_DRIVE_TRASH_ON_CLEANUP = bool(globals().get("EMPTY_GOOGLE_DRIVE_TRASH_ON_CLEANUP", True))
print("\nGoogle Drive trash cleanup:", "enabled" if EMPTY_GOOGLE_DRIVE_TRASH_ON_CLEANUP else "disabled")
drive_service = globals().get("drive_service", None)
if EMPTY_GOOGLE_DRIVE_TRASH_ON_CLEANUP:
    if drive_service is not None:
        print("   ✅ Drive API already authenticated")
    else:
        try:
            auth.authenticate_user()
            drive_service = build("drive", "v3")
            globals()["drive_service"] = drive_service
            print("   ✅ Drive API authenticated")
        except Exception as e:
            print(f"   Drive API auth failed: {e}")
            print("   Trash emptying disabled")
            drive_service = None
            globals()["drive_service"] = None
else:
    print("   Set EMPTY_GOOGLE_DRIVE_TRASH_ON_CLEANUP = True near the top to authorize Drive API Trash cleanup before training.")

def empty_drive_trash():
    """Permanently empty Google Drive Trash when explicitly enabled."""
    if not EMPTY_GOOGLE_DRIVE_TRASH_ON_CLEANUP:
        print("[empty_drive_trash] skipped; EMPTY_GOOGLE_DRIVE_TRASH_ON_CLEANUP=False")
        return False
    if drive_service is None:
        print("[empty_drive_trash] skipped; Drive API is not authenticated")
        return False
    print("[empty_drive_trash] permanently emptying Google Drive Trash...")
    drive_service.files().emptyTrash().execute()
    print("[empty_drive_trash] Google Drive Trash emptied. Google says storage accounting can take time to update.")
    return True

def extract_iteration(filename):
    match = CHECKPOINT_PATTERN.search(filename)
    return int(match.group(1)) if match else -1

def get_checkpoints_sorted(directory):
    if not os.path.exists(directory):
        return []
    ckpts = []
    for f in os.listdir(directory):
        it = extract_iteration(f)
        if it >= 0:
            ckpts.append((f, it))
    ckpts.sort(key=lambda x: x[1], reverse=True)
    return ckpts

def cleanup_directory(directory, keep_n=1):
    if not os.path.exists(directory):
        return 0
    deleted = 0
    ckpts = get_checkpoints_sorted(directory)
    for filename, iteration in ckpts[keep_n:]:
        try:
            os.remove(os.path.join(directory, filename))
            deleted += 1
            print(f"   Deleted: {filename}")
        except Exception:
            pass
    return deleted

def ensure_resume_checkpoint(directory):
    """Return SAM2's resume checkpoint, restoring checkpoint.pt if possible."""
    os.makedirs(directory, exist_ok=True)
    resume_path = os.path.join(directory, "checkpoint.pt")
    if os.path.exists(resume_path):
        return resume_path

    ckpts = get_checkpoints_sorted(directory)
    if ckpts:
        latest_name, latest_iter = ckpts[0]
        latest_path = os.path.join(directory, latest_name)
        if os.path.abspath(latest_path) != os.path.abspath(resume_path):
            shutil.copy2(latest_path, resume_path)
            print(f"   Restored checkpoint.pt from {latest_name} (iteration {latest_iter})")
        else:
            print(f"   checkpoint.pt already points to latest checkpoint (iteration {latest_iter})")
        return resume_path
    return None


def select_checkpoint_for_export(directory):
    best_val_path = os.path.join(directory, "checkpoint_best_val.pt")
    if os.path.exists(best_val_path) and os.path.getsize(best_val_path) > 0:
        return best_val_path, "checkpoint_best_val.pt", None
    resume_path = os.path.join(directory, "checkpoint.pt")
    if os.path.exists(resume_path) and os.path.getsize(resume_path) > 0:
        return resume_path, "checkpoint.pt", None
    ckpts = get_checkpoints_sorted(directory)
    if ckpts:
        latest_name, latest_iter = ckpts[0]
        return os.path.join(directory, latest_name), latest_name, latest_iter
    if os.path.exists(str(FINAL_CHECKPOINT_PATH)) and os.path.getsize(str(FINAL_CHECKPOINT_PATH)) > 0:
        return str(FINAL_CHECKPOINT_PATH), os.path.basename(str(FINAL_CHECKPOINT_PATH)), None
    return None, None, None


def export_final_checkpoint_from_training_dir(directory, require_checkpoint=False):
    latest_path, latest_name, latest_iter = select_checkpoint_for_export(directory)
    if latest_path is None:
        if require_checkpoint:
            raise FileNotFoundError(
                f"No SAM2 checkpoint found under {directory} or at {FINAL_CHECKPOINT_PATH}."
            )
        return None
    WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
    if os.path.abspath(latest_path) != os.path.abspath(str(FINAL_CHECKPOINT_PATH)):
        tmp_final = FINAL_CHECKPOINT_PATH.with_suffix('.tmp.pt')
        shutil.copy2(latest_path, tmp_final)
        os.replace(tmp_final, FINAL_CHECKPOINT_PATH)
        print(f"✅ Final checkpoint written to: {FINAL_CHECKPOINT_PATH}")
        print(f"   Source checkpoint: {latest_name}" + (f" (iteration {latest_iter})" if latest_iter is not None else ""))
    else:
        print(f"✅ Final checkpoint already available: {FINAL_CHECKPOINT_PATH}")
    best_val_metadata_path = os.path.join(directory, "checkpoint_best_val.json")
    if latest_name == "checkpoint_best_val.pt" and os.path.exists(best_val_metadata_path):
        with open(best_val_metadata_path) as _fh:
            best_val_metadata = json.load(_fh)
        print(f"   Best validation loss: {best_val_metadata.get('best_val_loss')} from {best_val_metadata.get('selection_metric')}")
    return {
        "source_checkpoint_path": latest_path,
        "source_checkpoint_name": latest_name,
        "source_checkpoint_iteration": latest_iter,
    }

if os.path.exists(OLD_BACKUP_DIR):
    print(f"   Removing checkpoint backup directory to free Drive space...")
    shutil.rmtree(OLD_BACKUP_DIR, ignore_errors=True)
    empty_drive_trash()
    print(f"   ✅ Checkpoint backup directory removed")

print("\nInitial checkpoint cleanup...")
resume_path = ensure_resume_checkpoint(CHECKPOINT_DIR)
num_deleted = cleanup_directory(CHECKPOINT_DIR, MAX_CHECKPOINTS_TO_KEEP)
if num_deleted > 0:
    print(f"   Deleted {num_deleted} old numbered checkpoint file(s)")
    empty_drive_trash()
ckpts = get_checkpoints_sorted(CHECKPOINT_DIR)
if resume_path:
    print(f"   ✅ SAM2 resume checkpoint: {resume_path}")
    if ckpts:
        print(f"   ✅ Latest numbered checkpoint: {ckpts[0][0]} (iteration {ckpts[0][1]})")
    print("   ✅ Training will auto-resume from checkpoint.pt")
else:
    print("   No checkpoints found - starting fresh")

_prelaunch_export = export_final_checkpoint_from_training_dir(CHECKPOINT_DIR, require_checkpoint=False)
if _prelaunch_export:
    print("   ✅ Existing checkpoint exported for inference before training launch")
if bool(globals().get("CHECKPOINT_RECOVERY_ONLY", False)):
    if _prelaunch_export is None:
        raise FileNotFoundError(
            f"CHECKPOINT_RECOVERY_ONLY=True, but no checkpoint exists under {CHECKPOINT_DIR} or at {FINAL_CHECKPOINT_PATH}."
        )
    print("CHECKPOINT_RECOVERY_ONLY=True: exported existing checkpoint; skipping SAM2 training launch.")
    raise SystemExit("Checkpoint recovery complete")

_cleanup_stop = threading.Event()

def background_cleanup_worker(directory, keep_n, interval=60):
    while not _cleanup_stop.is_set():
        _cleanup_stop.wait(interval)
        if _cleanup_stop.is_set():
            break
        try:
            n = cleanup_directory(directory, keep_n)
            if n > 0:
                empty_drive_trash()
        except Exception:
            pass

cleanup_thread = threading.Thread(
    target=background_cleanup_worker,
    args=(CHECKPOINT_DIR, MAX_CHECKPOINTS_TO_KEEP, 60),
    daemon=True
)
cleanup_thread.start()

print(f"\n   Background cleanup running every 60 seconds")
print("   Keeping checkpoint.pt plus checkpoint_best_val.pt when validation improves")
print(f"   Drive trash cleanup enabled: {EMPTY_GOOGLE_DRIVE_TRASH_ON_CLEANUP}")
print("\n✅ Checkpoint management active")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 8/8: Launch Training (subprocess instead of execvp)
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("STEP 8/8: LAUNCHING TRAINING")
print("=" * 80)
print("\n   Loss will print every 10 steps (log_freq: 10)")
print("   Checkpoint state saves every epoch to checkpoint.pt")
print("   Validation runs every epoch; checkpoint_best_val.pt is updated when validation loss improves")
print("   If checkpoint.pt exists, official SAM2 resumes model/optimizer/step state automatically")
print("   Cleanup runs every 60 seconds")
print("=" * 80)

time.sleep(3)

os.chdir("/content/segment-anything-2")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
# Reduce CUDA allocator fragmentation for borderline Colab GPU memory cases.
env.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

train_cmd = [
    "python", "training/train.py",
    "-c", CONFIG_NAME,
    "--use-cluster", "0",
    "--num-gpus", "1",
]

print(f"\nCommand: {' '.join(train_cmd)}\n")
print("=" * 80)
print("  TRAINING OUTPUT BELOW (loss appears every 10 steps)")
print("=" * 80 + "\n")

process = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=env,
    bufsize=1,
    universal_newlines=True,
    cwd="/content/segment-anything-2"
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    print("\n\nTraining interrupted by user")
    process.terminate()

return_code = process.wait()

_cleanup_stop.set()

print("\n" + "=" * 80)
if return_code == 0:
    print("  ✅ TRAINING COMPLETED SUCCESSFULLY")
else:
    print(f"  Training exited with code {return_code}")

num_deleted = cleanup_directory(CHECKPOINT_DIR, MAX_CHECKPOINTS_TO_KEEP)
if num_deleted > 0:
    empty_drive_trash()

resume_path = ensure_resume_checkpoint(CHECKPOINT_DIR)
ckpts = get_checkpoints_sorted(CHECKPOINT_DIR)
if ckpts:
    print(f"  Latest numbered checkpoint: {ckpts[0][0]} (iteration {ckpts[0][1]})")
elif resume_path:
    print(f"  Resume checkpoint available: {resume_path}")
print("=" * 80)

# Finalize the stable checkpoint path for inference. Prefer best validation, then resume, then latest numbered checkpoint.
_export_report = export_final_checkpoint_from_training_dir(CHECKPOINT_DIR, require_checkpoint=False)
if _export_report:
    # Keep checkpoint.pt and checkpoint_best_val.pt for resume and best-model auditability.
    cleanup_directory(CHECKPOINT_DIR, keep_n=0)
    ensure_resume_checkpoint(CHECKPOINT_DIR)
    for pt in WEIGHTS_DIR.glob('*.pt'):
        if pt.name != FINAL_CHECKPOINT_PATH.name:
            pt.unlink(missing_ok=True)
    empty_drive_trash()
else:
    print('⚠️ No checkpoint found to finalize.')


In [ ]:
# Final checkpoint recovery, verification, and inference manifest
import os
import re
import shutil
from pathlib import Path

SAM2_CONFIG_NAME = globals().get('SAM2_CONFIG_NAME', 'configs/sam2.1/sam2.1_hiera_l.yaml')
SAM2_BASE_CHECKPOINT_FILENAME = globals().get('SAM2_BASE_CHECKPOINT_FILENAME', 'sam2.1_hiera_large.pt')
SAM2_BASE_CHECKPOINT_URL = globals().get(
    'SAM2_BASE_CHECKPOINT_URL',
    'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt',
)

_CHECKPOINT_PATTERN = re.compile(r'checkpoint_(\d+)\.pt$')


def _numbered_checkpoints_sorted(directory: Path):
    if not directory.exists():
        return []
    items = []
    for path in directory.glob('checkpoint_*.pt'):
        match = _CHECKPOINT_PATTERN.match(path.name)
        if match:
            items.append((path, int(match.group(1))))
    items.sort(key=lambda item: item[1], reverse=True)
    return items


def _select_checkpoint_for_export(directory: Path):
    best = directory / 'checkpoint_best_val.pt'
    if best.exists() and best.stat().st_size > 0:
        return best, 'checkpoint_best_val.pt', None
    resume = directory / 'checkpoint.pt'
    if resume.exists() and resume.stat().st_size > 0:
        return resume, 'checkpoint.pt', None
    numbered = _numbered_checkpoints_sorted(directory)
    if numbered:
        path, iteration = numbered[0]
        return path, path.name, iteration
    if FINAL_CHECKPOINT_PATH.exists() and FINAL_CHECKPOINT_PATH.stat().st_size > 0:
        return FINAL_CHECKPOINT_PATH, FINAL_CHECKPOINT_PATH.name, None
    return None, None, None


def _export_checkpoint_for_inference():
    TRAINING_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    source_path, source_name, source_iteration = _select_checkpoint_for_export(TRAINING_CHECKPOINT_DIR)
    if source_path is None:
        raise FileNotFoundError(
            f'No SAM2 checkpoint found. Expected final checkpoint at {FINAL_CHECKPOINT_PATH} '
            f'or a training checkpoint under {TRAINING_CHECKPOINT_DIR}.'
        )
    WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
    if source_path.resolve() != FINAL_CHECKPOINT_PATH.resolve():
        tmp_final = FINAL_CHECKPOINT_PATH.with_suffix('.tmp.pt')
        shutil.copy2(source_path, tmp_final)
        os.replace(tmp_final, FINAL_CHECKPOINT_PATH)
        print(f'Exported checkpoint for inference: {source_name} -> {FINAL_CHECKPOINT_PATH}')
    else:
        print(f'Using existing final checkpoint: {FINAL_CHECKPOINT_PATH}')
    return {
        'source_checkpoint_path': str(source_path),
        'source_checkpoint_name': source_name,
        'source_checkpoint_iteration': source_iteration,
    }


export_report = _export_checkpoint_for_inference()
size_mb = FINAL_CHECKPOINT_PATH.stat().st_size / (1024 * 1024)
checkpoint_manifest = {
    'schema_version': 'syniscopy-sam2-checkpoint-manifest-v1',
    'notebook_version': RELEASE_NOTEBOOK_VERSION,
    'microscope_label': MICROSCOPE_LABEL,
    'final_checkpoint_path': str(FINAL_CHECKPOINT_PATH),
    'sam2_config_name': SAM2_CONFIG_NAME,
    'sam2_base_checkpoint_filename': SAM2_BASE_CHECKPOINT_FILENAME,
    'sam2_base_checkpoint_url': SAM2_BASE_CHECKPOINT_URL,
    'supervision_target': SUPERVISION_TARGET,
    'raw_dataset_root': str(RAW_DATASET_ROOT),
    'sam2_dataset_root': str(SAM2_DATASET_ROOT),
    'sam2_train_num_frames': int(SAM2_TRAIN_NUM_FRAMES),
    'sam2_max_num_objects': int(SAM2_MAX_NUM_OBJECTS),
    'sam2_reverse_time_prob': float(SAM2_REVERSE_TIME_PROB),
    'sam2_val_fraction': float(SAM2_VAL_FRACTION),
    'sam2_val_random_seed': int(SAM2_VAL_RANDOM_SEED),
    'sam2_loss_semantics_version': SAM2_LOSS_SEMANTICS_VERSION,
    'sam2_train_list_txt': str(SAM2_DATASET_ROOT / 'train_list.txt'),
    'sam2_val_list_txt': str(SAM2_DATASET_ROOT / 'val_list.txt'),
    'checkpoint_export': export_report,
}
_best_val_metadata = TRAINING_CHECKPOINT_DIR / 'checkpoint_best_val.json'
if _best_val_metadata.exists():
    with open(_best_val_metadata, 'r', encoding='utf-8') as _fh:
        checkpoint_manifest['best_validation'] = json.load(_fh)
CHECKPOINT_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(CHECKPOINT_MANIFEST_PATH, 'w', encoding='utf-8') as f:
    json.dump(checkpoint_manifest, f, indent=2)

print('Final checkpoint size MB:', f'{size_mb:.1f}')
print('Final checkpoint:', FINAL_CHECKPOINT_PATH)
print('Checkpoint manifest:', CHECKPOINT_MANIFEST_PATH)
